在仿真过程中，如果您希望某个参数（比如控制器增益、扰动强度、风速等）在不同的时间段取不同的值，可以通过以下几种常见的方法在 Python 代码中实现：

方法一：在主仿真循环中使用 if/elif/else 语句

这是最直接和常用的方法。在 simulation.py 的主循环内部，根据当前的仿真时间 t 来判断应该使用哪组参数值。

2.1 控制目标
对于轨迹跟踪问题，我们的控制目标是：

- 最小化实际轨迹与期望轨迹之间的误差
- 最小化控制输入的能量消耗
- 满足系统约束条件

In [ ]:
# 代价函数设计
J = sum(w_e * ||e1||^2 + w_v * ||e2||^2 + w_u * ||u||^2)

其中：

- e1 = y - yc (位置和姿态误差)
- e2 = y_dot - yc_dot (速度误差)
- u = tau (控制输入)
- w_e, w_v, w_u 为权重系数

In [ ]:
# simulation.py (部分 - 主循环内)
import numpy as np
import parameters as params # 假设参数初始值定义在这里
# ... 其他导入 ...

# --- 仿真循环 ---
# ... (初始化等代码) ...

for i, t in enumerate(sim_time):
    # --- 1. 根据时间调整参数 (Adjust parameters based on time) ---
    current_k1 = params.k1 # 默认使用 parameters.py 中的值
    current_disturbance_scale = 5000 # 默认扰动尺度
    current_wind = params.V_WIND_ERF # 默认风速

    if t < 50.0:
        # 时间段 1: 0 <= t < 50 秒
        # 可以保持默认值，或者设置特定的值
        print(f"时间 {t:.2f}s: 使用第一阶段参数")
        # current_k1 = np.array([0.05, 0.06, 0.05, 0.1, 0.1, 0.2]) # 示例：降低增益
    elif t < 120.0:
        # 时间段 2: 50 <= t < 120 秒
        print(f"时间 {t:.2f}s: 使用第二阶段参数")
        current_k1 = np.array([0.1, 0.12, 0.1, 0.3, 0.3, 0.5]) # 示例：提高增益
        current_disturbance_scale = 7000 # 示例：增大扰动
        current_wind = np.array([8.0, 3.0, 0.0]) # 示例：改变风速
    else:
        # 时间段 3: t >= 120 秒
        print(f"时间 {t:.2f}s: 使用第三阶段参数（可能恢复默认或使用新值）")
        # 可以恢复默认值
        current_k1 = params.k1
        current_disturbance_scale = 5000
        current_wind = params.V_WIND_ERF
        # 或者设置第三阶段的值
        # current_k1 = np.array([...])

    # --- 2. 在后续计算中使用调整后的参数 ---
    # 获取当前状态、期望状态等...

    # 更新扰动观测器 (如果需要调整观测器参数 l1-l5, beta1/2)
    # observer.l1 = new_l1_value # 如果需要修改
    # delta_hat = observer.update(...)

    # 计算控制输入 (使用调整后的增益 current_k1 等)
    # 确保 controller.calculate_control 使用的是 current_k1, k3, k4
    # 如果控制器内部直接引用 params.k1，需要修改控制器类允许传入增益
    # 或者在控制器类内部也加入时间判断
    # 假设 controller.calculate_control 可以接收增益作为参数 (推荐):
    # tau = controller.calculate_control(t, e1, e2, delta_hat, gamma, gamma_d, xc, xc_dot,
    #                                   k1=current_k1, k3=params.k3, k4=params.k4) # 传递调整后的k1

    # 获取实际扰动 (使用调整后的尺度)
    actual_delta = params.disturbance_delta(t, scale=current_disturbance_scale) # 需要修改 disturbance_delta 函数接受 scale 参数

    # 积分气艇模型 (如果风速变化，需要传递调整后的风速)
    def airship_ode(t_rk, X_rk):
        # ...
        # 获取当时的控制 tau (可能也基于调整后的增益)
        # 获取当时的扰动 d (可能基于调整后的尺度)
        current_delta_rk = params.disturbance_delta(t_rk, scale=current_disturbance_scale)
        # 获取当时的风速 wind_rk (可能基于调整后的风速)
        current_wind_rk = current_wind # 假设风速在小步长内不变
        # 修改 airship.rhs 让其能接收并使用变化的风速
        return airship.rhs(t_rk, X_rk, tau, lambda time_ignored: current_delta_rk, wind_erf=current_wind_rk) # 假设 rhs 能处理风速

    # ... (RK4 积分) ...

    # ... (记录数据) ...

实现此方法需要注意：
参数传递: 确保那些需要根据时间调整的参数（如 current_k1, current_disturbance_scale, current_wind）被正确地传递给使用它们的方法（如 controller.calculate_control, params.disturbance_delta, airship.rhs）。这可能需要修改这些方法的签名（增加参数）。
修改函数/方法:

可能需要修改 params.disturbance_delta 函数，让它接受一个可选的 scale 参数。

可能需要修改 airship.rhs 方法，让它接受一个 wind_erf 参数，并在计算相对速度时使用它，而不是直接从 self 或 params 读取。

可能需要修改 controller.calculate_control 方法，让它接受 k1, k3, k4 等增益作为参数，而不是在内部硬编码或直接读取 params。

方法二：定义时间分段函数

对于更复杂的参数变化逻辑，或者为了让主循环更简洁，可以定义一个函数来根据时间返回相应的参数值。

In [ ]:
# parameters.py (或一个单独的 parameter_schedule.py)

import numpy as np

def get_scheduled_params(t):
    """根据时间返回参数字典"""
    if t < 50.0:
        k1 = np.array([0.07, 0.08, 0.07, 0.2, 0.2, 0.4])
        disturbance_scale = 5000
        wind_erf = np.array([5.0, 2.0, 0.0])
    elif t < 120.0:
        k1 = np.array([0.1, 0.12, 0.1, 0.3, 0.3, 0.5])
        disturbance_scale = 7000
        wind_erf = np.array([8.0, 3.0, 0.0])
    else:
        k1 = np.array([0.07, 0.08, 0.07, 0.2, 0.2, 0.4]) # 恢复默认
        disturbance_scale = 5000
        wind_erf = np.array([5.0, 2.0, 0.0])

    # 返回包含所有时变参数的字典
    return {
        "k1": k1,
        "disturbance_scale": disturbance_scale,
        "wind_erf": wind_erf
        # 可以添加其他需要调度的参数
    }

# 可能还需要修改扰动函数以接受 scale
def disturbance_delta(t, scale=5000):
    """定义外部扰动向量，带有可调尺度"""
    d_vec = np.zeros(6)
    d_vec[0] = 0.5 + 2 * np.sin(0.1 * t)
    # ... (其他分量) ...
    d_vec[5] = 1.5 + 2 * np.cos(0.1 * t)
    return scale * d_vec

In [ ]:
# simulation.py (部分 - 主循环内)
# from parameter_schedule import get_scheduled_params # 如果单独存放
from parameters import get_scheduled_params, disturbance_delta # 假设放在 parameters.py

# ...

for i, t in enumerate(sim_time):
    # --- 1. 获取当前时间点的参数 ---
    scheduled_params = get_scheduled_params(t)
    current_k1 = scheduled_params["k1"]
    current_disturbance_scale = scheduled_params["disturbance_scale"]
    current_wind = scheduled_params["wind_erf"]

    # --- 2. 在后续计算中使用获取的参数 ---
    # ... (与方法一类似，确保参数正确传递) ...

    # 计算控制输入
    # tau = controller.calculate_control(..., k1=current_k1, ...)

    # 获取实际扰动
    actual_delta = disturbance_delta(t, scale=current_disturbance_scale)

    # 积分气艇模型
    def airship_ode(t_rk, X_rk):
        # ...
        current_delta_rk = disturbance_delta(t_rk, scale=current_disturbance_scale)
        current_wind_rk = current_wind
        return airship.rhs(t_rk, X_rk, tau, lambda time_ignored: current_delta_rk, wind_erf=current_wind_rk)
    # ... (RK4) ...

方法三：使用插值函数（例如 scipy.interpolate.interp1d）

如果参数需要在时间点之间平滑过渡，可以使用插值。

In [ ]:
# parameters.py (或 parameter_schedule.py)
import numpy as np
from scipy.interpolate import interp1d

# 定义参数变化的时间点和对应的值
time_points = np.array([0.0,  49.9, 50.0, 119.9, 120.0, params.T_SPAN]) # 注意包含过渡点
k1_values_at_points = np.array([
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4], # t=0
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4], # t=49.9 (保持第一阶段)
    [0.1, 0.12, 0.1, 0.3, 0.3, 0.5],   # t=50.0 (切换到第二阶段)
    [0.1, 0.12, 0.1, 0.3, 0.3, 0.5],   # t=119.9 (保持第二阶段)
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4], # t=120.0 (切换回默认)
    [0.07, 0.08, 0.07, 0.2, 0.2, 0.4]  # t=T_SPAN (保持默认)
])
disturbance_scale_values = np.array([5000, 5000, 7000, 7000, 5000, 5000])
# ... 对其他参数也这样做 ...

# 创建插值函数 (kind='previous' 实现阶跃, kind='linear' 实现线性插值)
# 注意：需要对每个参数或每个参数的分量单独创建插值函数
k1_interp_funcs = [interp1d(time_points, k1_values_at_points[:, i], kind='previous', bounds_error=False, fill_value=(k1_values_at_points[0, i], k1_values_at_points[-1, i])) for i in range(6)]
disturbance_scale_interp = interp1d(time_points, disturbance_scale_values, kind='previous', bounds_error=False, fill_value=(disturbance_scale_values[0], disturbance_scale_values[-1]))
# ... 对风速等其他参数也创建插值函数 ...

def get_interpolated_params(t):
    """使用插值函数获取参数"""
    current_k1 = np.array([f(t) for f in k1_interp_funcs])
    current_disturbance_scale = disturbance_scale_interp(t)
    # ... 获取其他插值参数 ...
    current_wind = params.V_WIND_ERF # 示例：风速不插值
    return {
        "k1": current_k1,
        "disturbance_scale": current_disturbance_scale,
        "wind_erf": current_wind
    }

In [ ]:
# simulation.py (部分 - 主循环内)
from parameters import get_interpolated_params, disturbance_delta

# ...
for i, t in enumerate(sim_time):
    # --- 1. 获取当前时间点的插值参数 ---
    interpolated_params = get_interpolated_params(t)
    current_k1 = interpolated_params["k1"]
    current_disturbance_scale = interpolated_params["disturbance_scale"]
    current_wind = interpolated_params["wind_erf"]

    # --- 2. 在后续计算中使用获取的参数 ---
    # ... (与之前类似，确保参数传递) ...

选择哪种方法？
方法一 (if/elif/else)： 最简单直观，适用于只有少数几个时间段和少量参数变化的情况。

方法二 (时间分段函数)： 代码更整洁，逻辑更清晰，适用于参数变化较多或逻辑较复杂的情况。推荐使用这种方法来组织代码。

方法三 (插值函数)： 适用于需要参数在时间点之间平滑过渡的场景，或者当你有大量时间点数据需要处理时。实现起来稍微复杂一些。

根据您的具体需求选择最合适的方法。对于控制器增益或扰动强度的阶跃变化，方法一 或 方法二 通常足够。

在aero_coefficients.py中为什么要设计图中的这部分代码？ 我想学习这种写法

好的，if __name__ == "__main__": 这个代码块在 Python 脚本中是一个非常常见且有用的设计模式。我们来详细学习一下它的作用和为什么要在 aero_coefficients.py 中包含它：

if __name__ == "__main__": 的含义

__name__ 是什么?

__name__ 是 Python 中一个内置的特殊变量。

它的值取决于你如何运行这个 Python 脚本文件：

直接运行: 如果你直接在命令行中执行这个脚本（例如 python aero_coefficients.py），那么在这个脚本内部，__name__ 的值会被自动设置为字符串 "__main__"。

作为模块导入: 如果这个脚本被其他 Python 文件通过 import aero_coefficients 的方式导入并使用，那么在这个脚本内部，__name__ 的值会被设置为这个模块的名字，也就是字符串 "aero_coefficients"。

if __name__ == "__main__": 的作用:

这行代码检查当前脚本是被直接运行还是被作为模块导入。

只有当脚本被直接运行时，if 条件 (__name__ == "__main__") 才为真，其下面的代码块才会被执行。

如果脚本是被导入的，if 条件为假，其下面的代码块就不会被执行。

为什么要在 aero_coefficients.py 中设计这部分代码？

在 aero_coefficients.py 中加入 if __name__ == "__main__": 代码块的主要目的是：

提供测试功能 (Testing):

aero_coefficients.py 文件定义了重要的计算函数（calculate_added_mass_inertia_local 和 get_aero_coefficients）。我们希望能够独立地测试这些函数是否能正常工作，计算结果是否符合预期，而不需要运行整个复杂的仿真 (simulation.py)。

通过将测试代码放在 if __name__ == "__main__": 块内，我们可以直接运行 python aero_coefficients.py 来执行这些测试。测试代码会：

调用 calculate_added_mass_inertia_local() 计算 k1, k2。

调用 get_aero_coefficients() 计算所有的气动系数。

打印出计算结果，方便开发者检查。

捕获可能发生的 ValueError 等异常，并打印错误信息。

模块化和可重用性 (Modularity & Reusability):

这个文件的主要目的是提供计算气动系数的功能（通过函数）给其他模块（如 simulation.py）使用。

当 simulation.py 执行 import aero_coefficients 时，它只需要导入文件顶部的参数定义和函数定义，不需要执行测试代码。if __name__ == "__main__": 确保了这一点。

这使得 aero_coefficients.py 既可以作为一个可独立测试的单元，又可以作为一个干净的功能模块被其他代码导入和重用。

避免副作用 (Avoiding Side Effects):

如果测试代码（比如 print 语句或函数调用）没有放在 if __name__ == "__main__": 块内，那么每次其他文件 import aero_coefficients 时，这些测试代码都会被执行，这通常是我们不希望发生的“副作用”。我们只希望在导入时获得定义，而不是执行测试。

总结:

if __name__ == "__main__": 是一种将“脚本作为程序直接运行时要执行的代码”（通常是测试、示例用法或主程序逻辑）与“脚本作为模块导入时要提供的定义”（函数、类、常量）分离开的标准 Python 写法。

在 aero_coefficients.py 中使用它，可以让你方便地：

直接运行该文件来测试气动系数的计算是否正确。

放心地在 simulation.py 中 import aero_coefficients 来使用其提供的函数和参数，而不会意外地执行测试代码。

这是一种非常好的编程实践，有助于提高代码的可测试性、模块化和可维护性。

In [ ]:
# 记得去看 notion关于部分的截图笔记 
import numpy as np
import matplotlib.pyplot as plt
from airship.trajectory import define_spiral_trajectory

def trajectory_tracking():
    # Time range for simulation
    t_values = np.linspace(0, 200, 1000)  # Simulate for 200 seconds with 1000 points
    dt = t_values[1] - t_values[0]  # Time step

    # Initialize actual state
    pos_actual = np.array([0.0, 0.0, 0.0])  # Initial position
    vel_actual = np.array([0.0, 0.0, 0.0])  # Initial velocity

    # Initialize arrays to store results
    pos_actual_history = []
    pos_desired_history = []

    # Get the trajectory function
    trajectory_function = define_spiral_trajectory(t_values)

    # PD Controller gains
    Kp = np.array([1.0, 1.0, 1.0])  # Proportional gain
    Kd = np.array([0.5, 0.5, 0.5])  # Derivative gain

    # Simulation loop
    for t in t_values:
        # Get desired trajectory
        pos_desired, vel_desired, acc_desired = trajectory_function(t)

        # Compute errors
        e_pos = pos_actual - pos_desired
        e_vel = vel_actual - vel_desired

        # Compute control input (acceleration)
        u = -Kp * e_pos - Kd * e_vel

        # Update actual state
        pos_actual += vel_actual * dt
        vel_actual += u * dt

        # Store results
        pos_actual_history.append(pos_actual.copy())
        pos_desired_history.append(pos_desired.copy())

    # Convert results to arrays
    pos_actual_history = np.array(pos_actual_history)
    pos_desired_history = np.array(pos_desired_history)

    # Plot results
    plt.figure(figsize=(8, 6))
    plt.plot(pos_desired_history[:, 0], pos_desired_history[:, 1], label="Desired Trajectory", color="blue")
    plt.plot(pos_actual_history[:, 0], pos_actual_history[:, 1], label="Actual Trajectory", color="red", linestyle="--")
    plt.xlabel("X Position (m)")
    plt.ylabel("Y Position (m)")
    plt.title("Trajectory Tracking in X-Y Plane")
    plt.legend()
    plt.grid()
    plt.show()

# Run the trajectory tracking simulation
trajectory_tracking()

In [ ]:
# main_1.py

import argparse
import logging
import sys

from simulation.run_simulation import run_simulation

def parse_args():
    p = argparse.ArgumentParser(
        description="Airship Simulation"
    )
    p.add_argument(
        "-m", "--mode",
        choices=["debug", "release"],
        default="release",
        help="仿真模式：debug 会打印更多日志，release 只打印 INFO+"
    )
    p.add_argument(
        "-l", "--log-file",
        type=str,
        default=None,
        help="如果提供，日志也会写到这个文件"
    )
    p.add_argument(
        "-t", "--duration",
        type=float,
        default=None,
        help="可选：覆盖配置里的仿真总时长 T_SPAN（单位 s）"
    )
    return p.parse_args()

def setup_logger(mode: str, log_file: str = None):
    """
    根据模式和可选的文件路径，配置 root logger。
    debug 模式：DEBUG 级别；release 模式：INFO 级别
    """
    level = logging.DEBUG if mode == "debug" else logging.INFO

    # 创建 handler 列表：屏幕输出 + （可选）文件输出
    handlers = [logging.StreamHandler(sys.stdout)]
    if log_file:
        handlers.append(logging.FileHandler(log_file, encoding="utf-8"))

    logging.basicConfig(
        level=level,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=handlers
    )

    logger = logging.getLogger("main")
    logger.debug(f"Logger set to {mode.upper()} (level={level})")
    if log_file:
        logger.info(f"日志也写入：{log_file}")
    return logger

def main():
    args = parse_args()
    logger = setup_logger(args.mode, args.log_file)
    logger.info("程序启动")

    # 如果用户指定了 duration，就动态覆盖 config.parameters.T_SPAN
    if args.duration is not None:
        import config.parameters as params
        logger.info(f"覆盖仿真总时长：{params.T_SPAN} → {args.duration}")
        params.T_SPAN = args.duration

    # 启动仿真
    # run_simulation() 内部会使用 config.parameters 里的 DT、T_SPAN、X0 等
    run_simulation()

    logger.info("仿真结束")

if __name__ == "__main__":
    main()



'''
下面是一个更“工业化”的 main.py 样例，演示如何一步步加入：
	1.	CLI 参数解析（用 argparse）
	2.	多种仿真模式（debug vs release）
	3.	日志同时输出到屏幕和文件

你可以把它直接复制到项目根目录的 main.py，并根据需要做微调。


⸻

如何逐步扩展
	•	更多 CLI 参数
	•	加 --dt 来覆盖步长 DT
	•	加 --output-dir 指定结果图 / 数据的输出目录
	•	多种仿真模式
	•	debug：日志等级 DEBUG，并且可以在 run_simulation() 里针对 mode=="debug" 打开更详细的可视化、加速器、断点等
	•	release：默认 INFO 级别，画图时使用更简洁的样式
	•	日志文件管理
	•	使用 RotatingFileHandler 每次运行切分日志
	•	在文件名里加时间戳：log-$(date).txt
	•	配置热加载
	•	用 yaml/json 读一套更复杂的实验配置，而不仅仅是 parameters.py

这样你的 main.py 就既简洁又灵活，可随项目成长不断迭代。

'''


'''
那段更“工业化”的 main.py 其实就像大多数命令行工具一样，先用 argparse 定义了一些可选参数，然后根据你传入的参数来配置日志、覆盖仿真时长、启动仿真。下面分步给你讲明白怎么用：

⸻

1. 查看帮助信息

在终端（或 VSCode/PyCharm 的 Terminal）里，切换到项目根目录后，输入：

python main.py --help

你会看到类似这样的输出：

usage: main.py [-h] [-m {debug,release}] [-l LOG_FILE] [-t DURATION]

Airship Simulation

optional arguments:
  -h, --help            show this help message and exit
  -m {debug,release}, --mode {debug,release}
                        仿真模式：debug 会打印更多日志，release 只打印 INFO+
  -l LOG_FILE, --log-file LOG_FILE
                        如果提供，日志也会写到这个文件
  -t DURATION, --duration DURATION
                        可选：覆盖配置里的仿真总时长 T_SPAN（单位 s）

这就是所有可用参数的说明。

⸻

2. 默认运行

如果你直接敲：

python main.py

	•	使用默认模式 release（只打印 INFO 和更高级别的日志到屏幕）
	•	不写入日志文件
	•	不覆盖脚本里 config.parameters.T_SPAN（仿真时长用你在 parameters.py 里写的值）

⸻

3. 使用 debug 模式

在开发或调试时，你可能想看更详细的日志（DEBUG 级别），就加上 -m debug：

python main.py -m debug

这样会把日志等级调到 DEBUG，屏幕上会输出非常详细的内部信息（方便排错）。

⸻

4. 同时输出到日志文件

如果你想把日志也写到文件里，带上 -l 参数：

python main.py -l simulation.log

这条命令会把所有日志（INFO 及以上）同时打印到屏幕和 simulation.log 文件。
你也可以跟 debug 模式一起用：

python main.py -m debug -l debug.log



⸻

5. 临时覆盖仿真时长

假设你在 config/parameters.py 里默认 T_SPAN = 200 秒，但这次想只跑 50 秒，就加上 -t 50：

python main.py -t 50

或者连同日志文件一起：

python main.py -t 50 -l short_run.log

脚本运行时会在日志里打印：

INFO  覆盖仿真总时长：200 -> 50



⸻

小结
	•	python main.py：默认运行
	•	-m/--mode：切换 release（默认）或 debug
	•	-l/--log-file：指定一个文件路径，将日志写入该文件
	•	-t/--duration：覆盖 parameters.py 中的 T_SPAN，以秒为单位

你可以根据需要自由组合这些选项，或者用 python main.py --help 随时查看。这样就能够“调”得动你的 main.py 了。祝仿真愉快！


'''

In [ ]:
# main_2.py

import argparse
import logging
import sys
import os
import json
from datetime import datetime
from logging.handlers import RotatingFileHandler

# 可选支持 yaml
try:
    import yaml
except ImportError:
    yaml = None

from simulation.run_simulation import run_simulation


def parse_args():
    p = argparse.ArgumentParser(description="Airship Simulation")
    p.add_argument(
        "-m", "--mode",
        choices=["debug", "release"],
        default="release",
        help="仿真模式：debug 会打印 DEBUG 日志，release 只打印 INFO+"
    )
    p.add_argument(
        "-l", "--log-file",
        type=str,
        default=None,
        help="日志写入文件，如果不指定，则只打印到控制台"
    )
    p.add_argument(
        "-c", "--config",
        type=str,
        default=None,
        help="可选：外部配置文件路径（.yaml 或 .json），用来覆盖 parameters.py 中的默认常量"
    )
    p.add_argument(
        "--dt",
        type=float,
        default=None,
        help="可选：覆盖仿真步长 DT（单位：秒）"
    )
    p.add_argument(
        "--output-dir",
        type=str,
        default=None,
        help="可选：仿真结果（图表/数据）的输出目录"
    )
    return p.parse_args()


def setup_logger(mode: str, log_file: str = None):
    """配置 root logger：console + （可选）文件输出 & RotatingFileHandler"""
    level = logging.DEBUG if mode == "debug" else logging.INFO

    # Console handler
    handlers = [logging.StreamHandler(sys.stdout)]

    # 如果指定了文件输出，就加一个滚动切分的 handler
    if log_file:
        # 在文件名里自动加上时间戳
        base, ext = os.path.splitext(log_file)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        fname = f"{base}_{timestamp}{ext or '.log'}"
        fh = RotatingFileHandler(
            fname,
            maxBytes=5 * 1024 * 1024,  # 5 MB 每个日志文件
            backupCount=3,             # 保留 3 份历史
            encoding="utf-8"
        )
        handlers.append(fh)

    logging.basicConfig(
        level=level,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=handlers
    )
    logger = logging.getLogger("main")
    logger.debug(f"Logger initialized in {mode.upper()} mode")
    if log_file:
        logger.info(f"Logging to file: {fh.baseFilename}")
    return logger


def apply_external_config(path: str, logger: logging.Logger):
    """读取 yaml/json 配置，动态写入到 config.parameters 模块"""
    if not os.path.isfile(path):
        logger.error(f"配置文件不存在：{path}")
        return

    ext = os.path.splitext(path)[1].lower()
    with open(path, "r", encoding="utf-8") as f:
        if ext in (".yaml", ".yml"):
            if yaml is None:
                logger.error("要使用 YAML 配置，需要先安装 pyyaml：pip install pyyaml")
                return
            cfg = yaml.safe_load(f)
        elif ext == ".json":
            cfg = json.load(f)
        else:
            logger.error("只支持 .yaml/.yml/.json 格式的配置文件")
            return

    import config.parameters as params
    for key, val in cfg.items():
        if hasattr(params, key):
            setattr(params, key, val)
            logger.info(f"参数覆盖: {key} = {val!r}")
        else:
            logger.warning(f"未知参数跳过: {key}")

    logger.info(f"外部配置 {path} 已加载")


def main():
    args = parse_args()

    # 1. 日志初始化
    logger = setup_logger(args.mode, args.log_file)
    logger.info("=== 程序启动 ===")

    # 2. 外部配置覆盖
    if args.config:
        apply_external_config(args.config, logger)

    # 3. 覆盖单个常量：DT
    if args.dt is not None:
        import config.parameters as params
        old = params.DT
        params.DT = args.dt
        logger.info(f"覆盖 DT: {old} -> {params.DT}")

    # 4. 创建输出目录（并写入 parameters）
    if args.output_dir:
        os.makedirs(args.output_dir, exist_ok=True)
        import config.parameters as params
        params.OUTPUT_DIR = args.output_dir  # run_simulation 里可读取
        logger.info(f"输出目录: {args.output_dir}")

    # 5. 启动仿真
    run_simulation()

    logger.info("=== 仿真结束 ===")


if __name__ == "__main__":
    main()



# 然后你这样在终端跑：
# python main.py --config settings.yaml



'''
下面是一个增强版的 main.py，演示如何一步到位实现：
	•	更多 CLI 参数：--dt、--output-dir、--config
	•	多种仿真模式：debug vs release
	•	日志同时输出到屏幕、文件，并使用滚动切分（RotatingFileHandler）
	•	热加载外部 yaml 或 json 配置，动态覆盖 config.parameters 中的常量

把它放到项目根目录（和 airship/、config/、simulation/ 同级），并确保安装了 pyyaml（若要读 .yaml）：



⸻

功能逐条解读
	1.	argparse 定义 CLI 接口
	•	--mode：切换 debug（DEBUG 级别）或 release（INFO 级别）。
	•	--log-file：日志写入文件，如果提供，则自动附加时间戳并滚动切分。
	•	--config：指定一个 .yaml 或 .json，用来一次性覆盖 config/parameters.py 里的默认值。
	•	--dt：只想临时改步长，就用这个。
	•	--output-dir：仿真生成的图、数据要存哪。
	2.	日志配置
	•	StreamHandler：把日志打到终端。
	•	RotatingFileHandler：如果指定了 --log-file，就把日志写到时间戳文件里，自动切分（5 MB/文件，最多保留 3 个历史）。
	3.	热加载外部配置
	•	根据文件后缀决定走 yaml.safe_load（需安装 PyYAML）还是 json.load。
	•	逐条覆盖 config.parameters 模块中已知的属性（其余跳过，并发出警告）。
	4.	单参数覆盖
	•	如果只想改步长，用 --dt，直接给 params.DT 赋新值。
	5.	输出目录
	•	自动创建目标文件夹并写入 params.OUTPUT_DIR，你的 run_simulation() 里可以读它，把图 plt.savefig(os.path.join(OUTPUT_DIR, ...))。
	6.	统一入口
	•	run_simulation() 本身不带参数，从模块里直接读 config.parameters。
	•	if __name__=="__main__" 确保：
	•	直接 python main.py 会跑仿真；
	•	导入 这个模块时（比如未来做单元测试），不会立刻执行仿真。

这样，你的 main.py 就灵活、可配置且“工业化”了：
	•	日常调试只要 python main.py -m debug
	•	生产环境可 python main.py -c my_settings.yaml --output-dir results/ -l app.log
	•	参数维护依旧在 config/parameters.py 或外部 config 文件里搞定。

'''



'''
三、注意事项
	1.	属性名要一一对应
	•	外部文件中的顶层键（key）要和 parameters.py 里那些常量名完全一致（区分大小写）。
	•	否则会被跳过并给你一个 warning。
	2.	数据类型要对
	•	YAML/JSON 里读出来的数字、列表、字典要和 parameters.py 对应属性原来的类型兼容。
	•	例如你在 parameters.py 定义 DT = 0.05（float），就别在 YAML 里写成字符串 "0.1"，而要写成 DT: 0.1（不带引号）。
	3.	导入时机
	•	一定要在“真正用参数”之前调用 apply_external_config()，否则仿真里拿到的还是老值。
	•	推荐在 main() 一开始、run_simulation() 之前完成覆盖。
	4.	YAML 支持可选
	•	如果项目没装 pyyaml，也能读 .json。装了再读 .yaml。
	•	安装：pip install pyyaml。
	5.	不要在模块顶层改
	•	一定首先 import config.parameters，然后在函数里用 setattr。这样只有当你主动调用 apply_external_config() 时才会改，不会影响到你直接在 REPL 或者单元测试时无意中改动。


'''

In [ ]:
""" 这个是model.py 底部做的笔记
'''  
# --- 可选的测试代码块在底部 ---
# 对于 airship 包下的这些模块文件，核心的类和函数定义必须放在顶层，以便能够被其他模块导入和使用。
# 可以选择性地在这个文件底部添加 if __name__ == "__main__": 块来包含仅仅用于独立测试该模块的代码。
# 绝对不要把主要的类和函数定义放到这个 if __name__ == "__main__": 块里面。
# 这允许您通过 python airship/model.py 这样的命令来单独测试 model.py 里的 Airship 类（如果需要的话），
# 但当它被 simulation 模块导入时，测试代码不会运行。

if __name__ == "__main__":
    # 这个块只在直接运行 python airship/model.py 时执行
    print("--- 测试 Airship 类 ---")
    # 创建一个初始状态 (可能需要从 config.parameters 导入)
    initial_X = np.zeros(12)
    initial_X[6] = 10 # 设置初始速度 u=10

    # 实例化 Airship
    test_ship = Airship(initial_X)
    print("Airship 对象已创建。")

    # 测试 rhs 方法 (提供假的 tau, disturbance, wind)
    current_t = 0.0
    fake_tau = np.zeros(6)
    def fake_dist(t): return np.zeros(6)
    fake_wind = np.array([1.0, 0, 0])

    try:
        dXdt = test_ship.rhs(current_t, test_ship.get_state(), fake_tau, fake_dist, fake_wind)
        print(f"rhs 方法在 t={current_t} 时计算得到的 dXdt:\n{dXdt}")
        assert dXdt.shape == (12,) # 检查输出形状
        print("rhs 方法基本运行测试通过。")
    except Exception as e:
        print(f"测试 rhs 时出错: {e}")

    # 可以添加更多针对特定方法的测试



'''


"""



In [ ]:
        #--- 计算总推力 (Calculate Total Thrust and  Torque) ---
        # --- 计算推力和推力力矩 (Calculate Thrust Forces and Moments) ---
        # 从tau中提取推力参数 / Extract thrust parameters from tau
        T_mag = tau[0]  # 推力大小 / Thrust magnitude
        mu = tau[1]     # 水平面内的推力偏转角 / Thrust deflection angle in the horizontal plane
        nu = tau[2]     # 垂直面内的推力偏转角 / Thrust deflection angle in the vertical plane

        # 计算右侧推力向量 / Calculate right thrust vector
        T_vec_r = np.array([
            [T_mag * np.cos(mu) * np.cos(nu)],
            [T_mag * np.sin(mu)],
            [T_mag * np.cos(mu) * np.sin(nu)]
        ])

        # 计算左侧推力向量 / Calculate left thrust vector
        T_vec_l = np.array([
            [T_mag * np.cos(mu) * np.cos(nu)],
            [T_mag * np.sin(mu)],
            [T_mag * np.cos(mu) * np.sin(nu)]
        ])

        # 计算总推力 /  Calculate total thrust
        T_total = T_vec_r + T_vec_l

        # 计算推力力矩 / Calculate thrust torque
        rp_r_1d = self.rp_r_vec.flatten()
        rp_l_1d = self.rp_l_vec.flatten()
        tau_r = np.cross(rp_r_1d, T_vec_r.flatten()).reshape(3, 1)
        tau_l = np.cross(rp_l_1d, T_vec_l.flatten()).reshape(3, 1)
        tau_vec = tau_r + tau_l       

In [ ]:
    @staticmethod
    def define_spiral_trajectory(t):
        """Define the desired spiral trajectory function """

        def trajectory(t):
            # 参数 parameters
            omega = 0.05  # 角速度  angular velocity  (rad/s)
            r = 100  # 基础半径  Basic radius (m)
            h_max = 150  # 最大高度 maximum altitude (m)

            # position pos(t)
            # 位置 pos(t)
            theta = omega * t
            pos = np.array([
                r * np.cos(theta),  # x
                r * np.sin(theta),  # y
                h_max * (1 - np.exp(-theta / 10))  # z：通过指数项渐进上升
            ])

            # velocity vel(t)
            # 速度 vel(t)
            vel = np.array([
                -r * omega * np.sin(theta),  # dx/dt
                r * omega * np.cos(theta),  # dy/dt
                h_max * (1 / 10) * np.exp(-theta / 10) * omega  # dz/dt
            ])

            # acceleration acc(t)
            # 加速度 acc(t)
            acc = np.array([
                -r * omega ** 2 * np.cos(theta),  # d^2x/dt^2
                -r * omega ** 2 * np.sin(theta),  # d^2y/dt^2
                -h_max * (1 / 100) * np.exp(-theta / 10) * omega ** 2  # d^2z/dt^2
            ])

            return pos, vel, acc

        return trajectory

In [ ]:
# ==============================================================================
#  (可选) 计算附加质量和惯性的函数 (Optional: Function to Calculate Added Mass/Inertia)
# ==============================================================================
# 这个函数也可以放在这里，或者放在单独的文件中  / This function can be placed here or in a separate file
def calculate_added_mass_inertia_local(a1=airship_a1, a2=airship_a2, b=airship_b, rho=rho_air_at_altitude):
    """
    在此文件内部计算附加质量和附加惯性矩阵 (仅用于演示)。
    Calculates added mass/inertia internally within this file (for demonstration).
    实际应用中，k1, k2 可能由外部计算并传入 get_aero_coefficients。
    In practice, k1, k2 might be calculated externally and passed to get_aero_coefficients.
    """
    # --- 重复附加质量计算逻辑 (Repeat added mass calculation logic) ---
    if b <= 0:
        raise ValueError("b must be positive")
    a = (a1 + a2) / 2.0
    if a <= 0:
        raise ValueError("a must be positive")
    if b > a:
        print(f"警告：扁椭球 b={b} > a={a}，附加质量公式可能不准确。")

    V = (4.0 / 3.0) * np.pi * a * b**2
    m_air = rho * V
    tolerance = 1e-9

    if abs(a - b) < tolerance:
        k1 = 0.5
        k2 = 0.5
        k3 = 0.5
    else:
        a_sq = a**2
        term_inside_sqrt = 1.0 - (b**2 / a_sq)
        if term_inside_sqrt < -tolerance:  # 允许小的负数容差 / Allow a small negative number tolerance
            print(f"警告：偏心率计算 sqrt 内部为负 ({term_inside_sqrt:.2e})，假设为球体。")
            k1 = 0.5
            k2 = 0.5
            k3 = 0.5
        else:
            term_inside_sqrt = max(0, term_inside_sqrt)  # 避免负数 / Avoid negative numbers
            e = np.sqrt(term_inside_sqrt)
            if abs(1.0 - e) < tolerance:
                raise ValueError("e is near 1.")
            e_sq = e**2

            if abs(e) < tolerance:  # 避免 f 和 g 中的除零 / Avoid division by zero in f and g
                # 接近球体的情况，用极限或直接设为球体值 / Near-spherical case, use limit or set to spherical value
                k1 = 0.5
                k2 = 0.5
                k3 = 0.5
            else:
                f_log = np.log((1.0 + e) / (1.0 - e))
                e_cubed = e**3
                if abs(e_cubed) < tolerance:
                    raise ValueError("e^3 near zero.")
                g = (1.0 - e_sq) / e_cubed
                alpha_prime = 2.0 * g * (f_log / 2.0 - e)
                if abs(e_sq) < tolerance:
                    raise ValueError("e^2 near zero.")
                beta_prime = (1.0 / e_sq) - (g * f_log / 2.0)

                denom_k1 = 2.0 - alpha_prime
                if abs(denom_k1) < tolerance:
                    raise ValueError("k1 denominator near zero.")
                k1 = alpha_prime / denom_k1

                denom_k2 = 2.0 - beta_prime
                if abs(denom_k2) < tolerance:
                    raise ValueError("k2 denominator near zero.")
                k2 = beta_prime / denom_k2

                b_sq = b**2
                term1_num_k3 = (b_sq - a_sq) * (alpha_prime - beta_prime)
                term2_den_k3 = 2.0 * (b_sq - a_sq) + (b_sq + a_sq) * (beta_prime - alpha_prime)
                if abs(term2_den_k3) < tolerance:
                    if abs(a - b) > tolerance:
                        raise ValueError("k3 denominator near zero (non-sphere).")
                    else:
                        k3 = 0.5
                else:
                    k3 = (1.0 / 5.0) * term1_num_k3 / term2_den_k3

    # --- 返回 k1, k2 (以及可能需要的 M', I0') / Return k1, k2 (and possibly M', I0') ---
    M_prime = m_air * np.diag([k1, k2, k2])
    I0_prime = m_air * np.diag([0.0, k3, k3])

    return k1, k2, M_prime, I0_prime

In [ ]:

# === 计算附加质量/惯性 (Calculate Added Mass/Inertia) ===
def calculate_added_mass_inertia(a1, a2, b, rho_air_):
    """
    根据双椭球体模型的几何参数计算附加质量和附加惯性矩阵。
    Calculates the added mass and added inertia matrices based on the
    geometric parameters of a double-ellipsoid model.

    参考公式来源：图片中提供的 Eq. 42 - 51
    Reference Equations: Eq. 42 - 51 from the provided image.

    Args:
        a1 (float): 第一个半长轴 (Semi-major axis 1).
        a2 (float): 第二个半长轴 (Semi-major axis 2).
        b (float): 半短轴 (Semi-minor axis).
        rho_air_ (float): 当地空气密度 (Local air density).

    Returns:
        tuple:包含两个 NumPy 数组的元组 (M_prime, I0_prime)
               A tuple containing two NumPy arrays: (M_prime, I0_prime)
               M_prime (np.ndarray): 附加质量矩阵 (Added mass matrix, 3x3).
               I0_prime (np.ndarray): 附加惯性矩阵 (Added inertia matrix, 3x3).

    Raises:
        ValueError: 如果几何参数无效 (If geometric parameters are invalid).
    """

    if b <= 0:
        raise ValueError("半短轴 b 必须大于 0 /Semi-minor axis b must be positive")

    # 计算平均半长轴 (Calculate mean semi-major axis 'a')
    a = (a1 + a2) / 2.0
    if a <= 0:
        raise ValueError("平均半长轴 a 必须大于 0 / Mean semi-major axis a must be positive")

    # 检查是否为长椭球 (Check for prolate spheroid assumption a >= b)
    # 注意：如果 b > a (扁椭球)，偏心率 e 和相关公式定义不同
    # Note: If b > a (oblate), eccentricity and related formulas differ.
    # 这里假设 a >= b，与图片中公式一致
    # Assuming a >= b as consistent with the provided formulas.
    if b > a:
        print(
            f"Warning: The current formula is applicable for prolate spheroids (a >= b), "
            f"but the input is a={a:.3f}, b={b:.3f} (oblate spheroid). "
            f"The result may be inaccurate."
        )
        # print(f"警告：当前公式适用于长椭球 (a >= b)，但输入为 a={a:.3f}, b={b:.3f} (扁椭球)。"
        #   f"结果可能不准确。")
        # 对于扁椭球需要不同的公式或检查源文献
        # Different formulas or source check needed for oblate case.
        # 为避免错误，可以抛出异常或继续计算（结果可能错误） /  Continue calculation (result may be incorrect)
        # raise ValueError("当前公式仅适用于 a >= b 的情况" / Current formula only for a >= b case")

    # 计算体积 (Calculate Volume V - Eq. 43)
    # V = (2.0 / 3.0) * np.pi * (a1 + a2) * b**2
    V = (4.0 / 3.0) * np.pi * a * b**2  # 使用平均值 a 的等效公式 Use equivalent formula with mean value a

    # 计算排开空气的质量 (Calculate mass of displaced air)
    m_air = rho_air_ * V

    # 处理特殊情况：球体 (Handle special case: Sphere)
    tolerance = 1e-9  # 定义一个小的容差 / Define a small tolerance
    if abs(a - b) < tolerance:
        # 对于球体 (For a sphere, a = b, e = 0)
        # k1_ = k2_ = k3_ = 0.5 (标准流体动力学结果 standard hydrodynamic result)
        k1_ = 0.5
        k2_ = 0.5
        k3_ = 0.5
    else:
        # 计算偏心率 (Calculate eccentricity e - Eq. 44)
        # 确保 ensure  a^2 > 0 且 and 1 - (b^2 / a^2) >= 0
        a_sq = a**2
        term_inside_sqrt = 1.0 - (b**2 / a_sq)
        if (1.0 - (b**2 / (a**2))) < 0:
            # 这理论上不应该在 a >= b 时发生，除非有数值误差
            print(f"警告：偏心率计算出现问题 / Eccentricity calculation warning " f"(term = {term_inside_sqrt:.2e})。将 e 设为 0 / set e as 0。")
            _e = 0.0
            k1_ = k2_ = k3_ = 0.5  # 退化为球体情况 / Fallback to sphere case
        else:
            _e = np.sqrt(1.0 - (b**2 / (a**2)))

            # 避免 e 极其接近 1 (避免 f 中的除零) / Avoid e close to 1 (to avoid division by zero in f)
            if abs(1.0 - _e) < tolerance:
                raise ValueError("偏心率 e 接近 1 (b 接近 0)，几何形状无效。/ Eccentricity e approaches 1 (invalid geometry)")

            # 计算中间参数 / Calculate intermediate parameters f, g, alpha_prime, beta_prime
            # Calculate intermediate parameters f, g, alpha_prime, beta_prime

            # f (Eq. 45)
            f = np.log((1.0 + _e) / (1.0 - _e))

            # g (Eq. 46)
            # 避免 e=0 (已在球体情况中处理) / Avoid division by zero for e=0 (handled in sphere case)
            e_sq = _e**2
            e_cubed = _e**3
            if abs(e_cubed) < tolerance:
                # 理论上 e 非零，但数值上可能很小 / Theoretically e is non-zero, but numerically small
                raise ValueError("偏心率 e 的立方接近于零，无法计算 g。 / Eccentricity e cubed is close to zero, cannot calculate g.")
            _g = (1.0 - e_sq) / e_cubed

            # alpha_prime (Eq. 47)
            alpha_prime = 2.0 * _g * (f / 2.0 - _e)

            # beta_prime (Eq. 48)
            if abs(e_sq) < tolerance:
                raise ValueError(
                    "偏心率 e 的平方接近于零，无法计算 beta_prime。" "The square of eccentricity e is close to zero, beta_prime cannot be calculated."
                )
            beta_prime = (1.0 / e_sq) - (_g * f / 2.0)

            # 计算惯性因子 k1, k2, k3 / Calculate inertia factors k1, k2, k3
            # k1 (Eq. 49)
            denominator_k1 = 2.0 - alpha_prime  # denominator 分母   numer 分子，，fraction 分数
            if abs(denominator_k1) < tolerance:
                raise ValueError("计算 k1 时分母接近零。/ Small denominator in k1 calculation")
            k1_ = -alpha_prime / (2.0 - alpha_prime)

            # k2 (Eq. 50)
            denominator_k2 = 2.0 - beta_prime
            if abs(denominator_k2) < tolerance:
                raise ValueError("计算 k2 时分母接近零。/ Small denominator in k2 calculation")
            k2_ = -beta_prime / (2.0 - beta_prime)

            # k3 (Eq. 51)
            a_sq = a**2
            b_sq = b**2
            term1_num_k3 = (b_sq - a_sq) * (alpha_prime - beta_prime)
            term2_den_k3 = 2.0 * (b_sq - a_sq) + (b_sq + a_sq) * (beta_prime - alpha_prime)

            if abs(term2_den_k3) < tolerance:
                # 检查球体情况是否已处理 (e=0 -> a=b -> b^2-a^2 = 0)
                # / Check if sphere case has been handled (e=0 -> a=b -> b^2-a^2 = 0)
                # 如果 a != b 但分母为零，表示可能有其他问题或特殊共振情况
                # / If a != b but denominator is zero, there may be another issue or resonance case.
                if abs(a - b) > tolerance:
                    raise ValueError("计算 k3 时分母接近零 (非球体情况) Small denominator in k3 calculation。")
                else:  # 如果是球体，分子也为零，极限应为 0.5 / If it's a sphere, numerator is also zero, limit should be 0.5
                    k3_ = 0.5
            else:
                k3_ = -(1.0 / 5.0) * term1_num_k3 / term2_den_k3

    # 构建附加质量矩阵 (Construct Added Mass Matrix M_prime - Eq. 42)
    _M_prime = m_air * np.diag([k1_, k2_, k2_])

    # 构建附加惯性矩阵 (Construct Added Inertia Matrix I0' - Eq. 42)
    _I0_prime = m_air * np.diag([0.0, k3_, k3_])  # 注意第一个元素是 0 /note first element is 0

    return _M_prime, _I0_prime

In [ ]:
# 这里的U_ref每个仿真时刻都用同一个值，是不是不合理，且不对，教我怎么改正确来，具体要改到哪些内容，教我
#基于轨迹特性计算参考控制
# 使用上一次的优化结果
# 在控制器类中添加属性存储上一次的控制序列，然后在这里使用：
"""
# 修改 NMPCThrustController 类，添加属性
self.last_optimal_sequence = None

# 修改 step 方法结尾部分，保存整个最优控制序列
u_sequence = []
for i in range(N):
    u_i = w_opt["x"].full().flatten()[(i+1)*12+i*3:(i+1)*12+(i+1)*3]
    u_sequence.append(u_i)
self.last_optimal_sequence = u_sequence

# 然后在 run_nmpc_simulation 中使用
if controller.last_optimal_sequence is not None:
    # 使用前一时刻的最优控制序列后移一步
    for j in range(params.N_HORIZON-1):
        U_ref.append(controller.last_optimal_sequence[j+1])
    # 对最后一步重复使用最后的控制
    U_ref.append(controller.last_optimal_sequence[-1])
else:
    # 首次运行时使用简单猜测
    for j in range(params.N_HORIZON):
        U_ref.append(np.array([8.0, 0.0, 0.0]))

"""

In [ ]:
#main.py





"""
# main.py

"""
import sys
import os
import logging
import traceback
from datetime import datetime
from simulation.run_simulation import run_simulation, run_nmpc_simulation



# 添加项目根目录到 sys.path / Add project root directory to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)





def setup_logger():
    """
    全局日志配置：只需在这里配置一次，项目中其他模块获取同一 logger 即可。
    Global logging configuration: Configure once here, and other modules in the project
    can retrieve the same logger.
    """
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True)  # 创建 logs 目录 / Create logs directory if it doesn't exist
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(log_dir, f"simulation_{timestamp}.log")

    logging.basicConfig(
        level=logging.DEBUG,  # 可以调整级别 / Adjust the level as needed
        format="%(asctime)s - %(levelname)s - %(message)s",
        handlers=[logging.FileHandler(log_file, mode="w"), logging.StreamHandler()],  # Also output to terminal
    )
    return logging.getLogger(__name__)




def main():
    """
    主函数
    """
    logger = setup_logger()
    logger.info("程序启动 / Program started")

    # ==== 控制仿真模式和轨迹类型（手动切换）/ Control simulation mode and trajectory type ====
    simulation_mode = "nmpc"  # 控制器选择："blf" 或 "nmpc"
    trajectory_type = "linear"  # 轨迹选择："default", "spiral", "figure8", "lemniscate", "linear"
    use_disturbance_compensation = True  # 是否使用扰动补偿

    try:
        if simulation_mode == "blf":
            run_simulation(trajectory_type=trajectory_type)
        elif simulation_mode == "nmpc":
            run_nmpc_simulation(use_disturbance_compensation=use_disturbance_compensation,
                                trajectory_type=trajectory_type)
        else:
            raise ValueError(f"未知的仿真模式 / unknown simulation mode: {simulation_mode}")
    except (ValueError, RuntimeError, TypeError, AttributeError) as e:
        logger.error("仿真过程中发生错误：%s", e)
        traceback.print_exc()
    else:
        logger.info("仿真成功完成 / Simulation completed successfully")  # 3. 结束日志 / End logging


if __name__ == "__main__":
    print("[INFO]: starting information ")
    main()


In [ ]:
"""
observer.py
Fixed-time Disturbance Observer (DO) for Airship
Disturbance observer module, supporting disturbance compensation under the NMPC control strategy.
"""


# cspell:ignore R_block coeff
# pylint: disable=invalid-name




import numpy as np
import casadi as ca
from config import parameters as params
from .utils import R_block



# Modified NMPCDisturbanceObserver class in airship/observer.py

class NMPCDisturbanceObserver:
    """
    Estimate the disturbance: by observing the state error of the airship (position error, velocity error, etc.), estimate the magnitude and direction of the external disturbance.
    Compensate the disturbance: feed the estimated disturbance value back to the controller, used to compensate the influence of the external disturbance on the airship trajectory tracking.
    Support NMPC controller: provide a symbolic disturbance observer equation for the prediction model of the NMPC controller.
    A disturbance observer specifically designed for the NMPC controller, using CasADi symbolic calculation
    """
    def __init__(self):
        # Basic observer parameters (obtained from the params module or using default values)
        self.l1 = params.l1 if hasattr(params, 'l1') else 2.0
        self.l2 = params.l2 if hasattr(params, 'l2') else 1.0
        self.l3 = params.l3 if hasattr(params, 'l3') else 1.5
        self.l4 = params.l4 if hasattr(params, 'l4') else 2.0
        self.l5 = params.l5 if hasattr(params, 'l5') else 1.0
        self.beta1 = params.beta1 if hasattr(params, 'beta1') else 0.5
        self.beta2 = params.beta2 if hasattr(params, 'beta2') else 1.5
        self.M = params.M_cfg
        self.M_inv = params.M_inv

        # Initialize the observer state
        self.z1_hat = np.zeros(6) # Estimation of position and velocity errors
        self.e2_hat = np.zeros(6) # Estimation of velocity errors
        self.delta_hat = np.zeros(6) # Estimation of disturbance

        # Parameters for disturbance filtering
        self.filter_coeff = params.do_filter_coeff if hasattr(params, 'do_filter_coeff') else 0.7
        self.prev_delta_hat = np.zeros(6)

        # Disturbance compensation gain
        self.compensation_gain = params.do_compensation_gain if hasattr(params, 'do_compensation_gain') else 0.9

        # History
        self.history = []

        # Create a CasADi symbolic function version, used for NMPC prediction
        self._create_symbolic_observer()

    def _create_symbolic_observer(self):
        """Create a CasADi symbolic version of the observer equation, used for the prediction model of the NMPC controller
        Note: This function does not return a value because it constructs a CasADi symbolic function
        observer_update_func, and binds it to the class variable self.observer_update_func,
        for other methods (such as update()) to call.
        
        Args:
            None
        Returns:
            None
        """
        # Define symbolic variables
        # __import__() is Python's underlying function for dynamic module import, equivalent to import casadi as ca
        # ca = __import__("casadi") # Import casadi library
        e1_sym = ca.SX.sym("e1", 6) # Position/attitude error vector
        e2_sym = ca.SX.sym("e2", 6) # Velocity and angular velocity error
        tau_sym = ca.SX.sym("tau", 6) # Control input (force and torque)
        gamma_sym = ca.SX.sym("gamma", 3) # Attitude angle (Euler angle)
        dt_sym = ca.SX.sym("dt", 1) # Time step
        z1_hat_sym = ca.SX.sym("z1_hat", 6) # Internal state of the observer
        e2_hat_sym = ca.SX.sym("e2_hat", 6) # Internal state of the observer

        # Build the observer equation
        R_sym = R_block(gamma_sym)  # Assume R_block already supports CasADi symbolic
        RM_inv_sym = R_sym @ self.M_inv # Combination of rotation matrix and mass matrix, used to convert the control input to the rate of change of velocity error

        # e2_hat update
        e2_hat_dot_sym = -self.l1 * e2_hat_sym + RM_inv_sym @ tau_sym
        e2_hat_next_sym = e2_hat_sym + e2_hat_dot_sym * dt_sym

        # z1 and z2 calculation
        z1_sym = e2_sym - e2_hat_sym
        z2_sym = self.l2 * z1_sym

        # z1_hat update (full version, including nonlinear terms)
        # Define the symbolic version of the sig function
        def sig_sym(x, alpha):
            """
            Symbolic version of the sig function, used to replace np.sign(x) * (np.abs(x) ** alpha)
            sig(x, alpha) = sign(x) * |x|^alpha
            Compute signed power function, used to construct nonlinear terms (such as sliding mode control, etc.)
            
            Args:
                x: Input variable
                alpha: Power exponent

            Returns:
                Signed power result
            """
            return ca.sign(x) * (ca.fabs(x) ** alpha)  # np.pow(ca.fabs(x), alpha)

        # z1_hat update
        z1_hat_dot_sym = (-self.l1 * z1_hat_sym + z1_sym + self.l3 * z2_sym +
                          self.l4 * sig_sym(z1_hat_sym, self.beta1) +
                          self.l5 * sig_sym(z1_hat_sym, self.beta2))
        z1_hat_next_sym = z1_hat_sym + z1_hat_dot_sym * dt_sym

        # Calculate the disturbance estimate
        delta_star_hat_sym = (z2_sym + self.l1 * self.l2 * z1_hat_sym) / self.l2
        # f_term is usually handled internally by the controller in the NMPC model
        f_term_sym = ca.SX.zeros(6)  # Simplified processing, provided by the controller

        # Calculate the disturbance estimate
        delta_hat_raw_sym = self.M @ ca.transpose(R_sym) @ (delta_star_hat_sym - self.l1 * e2_sym - f_term_sym)

        # Create the update function, generate the symbolic function
        self.observer_update_func = ca.Function(
            'observer_update',
            [e1_sym, e2_sym, tau_sym, gamma_sym, dt_sym, z1_hat_sym, e2_hat_sym],
            [z1_hat_next_sym, e2_hat_next_sym, delta_hat_raw_sym],
            ['e1', 'e2', 'tau', 'gamma', 'dt', 'z1_hat', 'e2_hat'],
            ['z1_hat_next', 'e2_hat_next', 'delta_hat_raw']
        )

    def update(self, dt, e1, e2, tau, gamma, f_func=None):
        """
        Update the disturbance estimate

        Parameters:
            dt: Time step
            e1: Position/attitude error vector
            e2: Velocity/angular velocity error vector
            tau: Control input (force and torque)
            gamma: Attitude angle (Euler angle)
            f_func: Function to calculate f(e1,e2) (optional)

        Returns:
            delta_hat_compensated: Disturbance vector that can be directly used for compensation
        """
        # Update the internal state
        z1_hat_next, e2_hat_next, delta_hat_raw = self.observer_update_func(
            e1, e2, tau, gamma, dt, self.z1_hat, self.e2_hat
        )

        self.z1_hat = z1_hat_next
        self.e2_hat = e2_hat_next

        # If f_func is provided, use it to calculate f_term
        if f_func is not None:
            # Calculate the rotation matrix
            R = R_block(gamma)
            z1 = e2 - self.e2_hat
            z2 = self.l2 * z1
            delta_star_hat = (z2 + self.l1 * self.l2 * self.z1_hat) / self.l2

            # Calculate the f(e1, e2) term
            f_term = f_func(e1, e2)

            # Recalculate the disturbance estimate (considering f_term)
            delta_hat_raw = self.M @ R.T @ (delta_star_hat - self.l1 * e2 - f_term)

        # Apply low-pass filter
        self.delta_hat = self.filter_coeff * self.prev_delta_hat + (1 - self.filter_coeff) * delta_hat_raw
        self.prev_delta_hat = self.delta_hat

        # Apply compensation gain / Disturbance estimate that can be directly used for compensation
        delta_hat_compensated = self.delta_hat * self.compensation_gain

        # Record history
        self.history.append({
            'delta_hat_raw': np.array(delta_hat_raw).flatten(),
            'delta_hat_filtered': self.delta_hat.full().flatten(),
            'delta_hat_compensated': delta_hat_compensated.full().flatten()
        })

        return delta_hat_compensated

    def get_current_disturbance_estimate(self):
        """Return the current disturbance estimate

        Returns:
            Current disturbance estimate
        """
        return self.delta_hat

    def get_compensated_estimate(self):
        """Return the disturbance estimate considering the compensation gain
        """
        return self.delta_hat * self.compensation_gain

    def reset(self):
        """Reset the observer state
        Used to reset the state variables and history of the observer
        """
        self.z1_hat = np.zeros(6)
        self.e2_hat = np.zeros(6)
        self.delta_hat = np.zeros(6)
        self.prev_delta_hat = np.zeros(6)
        self.history = []


In [ ]:
# model.py
"""
Airship dynamic model module (model.py)
"""
# pylint: disable=invalid-name
# cspell:ignore coeffs ddelta eta_f Sh Sg Sf Cdcf dalpha arcsin coeff ndarray linalg vertcat xdot
# cspell:ignore arctan RUDT RUDB ELVL ELVR unmodeled
# === Standard Libraries ===
import sys
import os

# === Third-party Libraries ===
import numpy as np

import casadi as ca

# === Local Modules ===
from config import parameters as params
from airship.aero_force_torque import calculate_aero_forces_moments, calculate_relative_velocity, calculate_aoa_sideslip
from airship.thrust import thrust_params_to_force_torque
from airship.utils import skew, R_zeta, R_block



# === Set Path (if needed) ===
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))




class Airship:
    """
    Airship model class
    """

    def __init__(self, initial_state):
        self.X = initial_state  # [zeta, gamma, v, omega] (12x1)
        self.M = params.M_cfg  # Combined inertia matrix from Eq. 9
        self.M_inv = params.M_inv
        self.m = params.m  # Mass
        self.g = params.g  # Gravity
        self.I0 = params.I0  # Inertia matrix I0
        self.M_upper_left = params.M_cfg[0:3, 0:3]  # m*I + M' from Eq. 9
        self.rc_vec = params.rc  # Vector CV->CG (shape (3,1))
        self.rb_vec = params.rb  # Vector CV->CB (shape (3,1))
        self.rp_r_vec = params.rp_r  # Right thrust application point vector (Vector CV->CP Right)
        self.rp_l_vec = params.rp_l  # Left thrust application point vector (Vector CV->CP Left)
        self.rc_skew = skew(self.rc_vec.flatten())  # Skew-symmetric matrix for rc

        # --- Add parameters for buoyancy/aero ---
        self.Vol_airship = params.Vol_airship  # Volume
        self.rho_air = params.rho_air  # Air density
        self.S_ref = params.S_ref  # Reference Area
        self.L_ref = params.L_ref  # Reference Length

        # Add reference to wind speed parameters
        self.V_wind_erf_const = params.V_WIND_ERF  # if wind speed is constant
        # self.V_wind_func = params.V_WIND_FUNC # if wind speed is a function


    def rhs(self, t, X, tau, disturbance_func):
        """Calculate the derivative of the state vector X - Right Hand Side"""
        _zeta = X[0:3]
        gamma = X[3:6]
        v = X[6:9]  # Linear velocity in BRF [u, v, w]
        omega = X[9:12]  # Angular velocity in BRF [p, q, r]
        x_vec = X[6:12]  # Combined velocity state [v, omega]

        # Ensure v and omega are 1D arrays for calculations
        v_1d = v.flatten()
        omega_1d = omega.flatten()
        rc_1d = self.rc_vec.flatten()  # Vector from CG to CV
        rb_1d = self.rb_vec.flatten()  # Vector from CB to CV
        u, v_body, w = v_1d[0], v_1d[1], v_1d[2]
        p, q, r = omega_1d[0], omega_1d[1], omega_1d[2]

        # --- Kinematics - Eq. 5 / Eq. 12 first part ---
        # Note: Kinematic equations describe the motion of the airship relative to the ground
        R = R_block(gamma)  # Combined rotation matrix diag(R_zeta, R_y)
        y_dot = R @ x_vec  # [zeta_dot, gamma_dot]  (relative to ground)


        # --- Dynamics - Eq. 8 / Eq. 12 second part ---

        # --- Calculate N term - Eq. 10 ---
        # N = [ N1 ; N2 ] where N1 is 3x1 (forces) and N2 is 3x1 (torques)
        _v_col = v.reshape(3, 1)
        omega_col = omega.reshape(3, 1)
        rc_col = self.rc_vec  # Already (3,1)  # Vector from CG to CV

        # Calculate common cross products
        omega_cross_v = np.cross(omega, v, axis=0).reshape(3, 1)  # omega x v
        omega_cross_rc = np.cross(omega, rc_col.flatten(), axis=0).reshape(3, 1)  # omega x rc
        omega_cross_omega_cross_rc = np.cross(omega, omega_cross_rc.flatten(), axis=0).reshape(3, 1)  # omega x (omega x rc)
        omega_cross_I0_omega = np.cross(omega, (self.I0 @ omega_col).flatten(), axis=0).reshape(3, 1)  # omega x (I0*omega)
        rc_cross_omega_cross_v = np.cross(rc_col.flatten(), omega_cross_v.flatten(), axis=0).reshape(3, 1)  # rc x (omega x v)

        # N1 = (m*I + M') * (omega x v) + m * omega x (omega x rc)
        # Note: M_upper_left already contains (m*I + M')
        N1 = self.M_upper_left @ omega_cross_v + self.m * omega_cross_omega_cross_rc

        # N2 = omega x (I0*omega) + m*rc x (omega x v)
        N2 = omega_cross_I0_omega + self.m * rc_cross_omega_cross_v

        N_term = np.vstack((N1, N2)).flatten()  # Combine N1 and N2 into 6x1 vector

        # === Calculate F term - Eq. 11 ===
        # F = [ F_forces ; F_torques ]
        # F_forces = fg - fb + fa
        # F_torques = mg + mb + ma

        # === Calculate Gravity Force and Torque ---
        # fg: Gravity force in BRF
        Rz = R_zeta(gamma)  # Rotation from BRF to ERF
        gravity_ERF = np.array([[0], [0], [self.m * self.g]])  # Gravity in Earth Frame
        fg_BRF = Rz.T @ gravity_ERF  # Rotate gravity vector to Body Frame

        # mg: Gravity torque in BRF
        # Torque = r_cg x F_g.  Since F_g acts at CG, r_cg = 0.
        mg_BRF = np.cross(rc_1d, fg_BRF.flatten()).reshape(3, 1)  # Torque due to gravity acting at CG (rc is CV->CG)

        # === Calculate Buoyancy Force and Torque ---
        # fb: Buoyancy force in BRF
        # Requires displaced volume V and air density rho_air.
        F_buoyancy_ERF = np.array([[0], [0], [-self.Vol_airship * self.rho_air * self.g]])  # upward is negative Z
        fb_BRF = Rz.T @ F_buoyancy_ERF  # Rotate buoyancy vector to Body Frame

        # mb: Buoyancy torque in BRF
        # Torque = r_cb x F_b. r_cb = vector from CV to CB. Assume r_cb = -rc_vec.
        # Requires fb_BRF which is assumed zero here.
        # Torque due to buoyancy acting at CB (assumed at CV, so arm is -rb)
        mb_BRF = np.cross(-rb_1d, fb_BRF.flatten(), axis=0).reshape(3, 1)
        # If fb_BRF was non-zero:
        # r_cb = -self.rb_vec
        # mb_BRF = np.cross(r_cb.flatten(), fb_BRF.flatten(), axis=0).reshape(3,1)

        # === New: Calculate Relative Velocity ---
        # Get wind velocity
        # if using constant wind speed:
        V_wind_ERF = self.V_wind_erf_const
        # if using function wind speed:
        # V_wind_ERF = self.V_wind_func(t, zeta)

        # Transform wind to Body Frame
        V_wind_BRF = Rz.T @ V_wind_ERF.reshape(3, 1)
        V_wind_BRF_1d = V_wind_BRF.flatten()

        # Calculate relative velocity - relative Airspeed in BRF
        v_airship_brf_1d = v_1d  # Body Reference Frame - BRF
        v_rel_brf_1d, u_rel, v_rel_body, w_rel = calculate_relative_velocity(v_airship_brf_1d, V_wind_BRF_1d)


        # === Calculate Aerodynamic Forces and Moments ---
        # fa: Aerodynamic force in BRF  placeholder
        # Dynamic Pressure  - based on relative speed
        V_rel_mag = np.linalg.norm(v_rel_brf_1d)
        q_dyn = 0.5 * self.rho_air * V_rel_mag**2 if V_rel_mag > 1e-3 else 0  # Avoid division by zero

        # Angle of Attack & Sideslip Angle - based on relative speed
        # Ensure u > 0
        alpha, beta = calculate_aoa_sideslip(u_rel, v_rel_body, w_rel, V_rel_mag)

        # --- Get Control Surface Deflections ---
        #  Critical Placeholder: These values need to be determined by a Control Allocation module based on tau[3:6] !!!
        delta_RUDT = np.deg2rad(0.0)  # [rad] - Placeholder
        delta_RUDB = np.deg2rad(0.0)  # [rad] - Placeholder
        delta_ELVL = np.deg2rad(0.0)  # [rad] - Placeholder
        delta_ELVR = np.deg2rad(0.0)  # [rad] - Placeholder

        # --- Calculate Aerodynamic Forces and Moments ---
        fa_BRF, ma_BRF = calculate_aero_forces_moments(
            q_dyn, alpha, beta,
            params.AERO_COEFFS,
            delta_RUDT, delta_RUDB, delta_ELVL, delta_ELVR
        )

        # === Calculate Thrust and torque ---
        print(type(tau)) # Ensure tau is a NumPy array
        print(tau)
        # Use thrust_params_to_force_torque to convert thrust parameters to thrust and torque
        thrust_torque = thrust_params_to_force_torque(tau, self.rp_r_vec, self.rp_l_vec)
        print(thrust_torque)

        T_total = thrust_torque[0:3].reshape(3, 1)  # Thrust vector - Thrust vector
        tau_vec = thrust_torque[3:6].reshape(3, 1)  # Torque vector - Torque vector

        # === Combine Forces and Torques ---
        F_forces = fg_BRF - fb_BRF + fa_BRF + T_total
        print(F_forces.shape)
        F_torques = mg_BRF + mb_BRF + ma_BRF + tau_vec
        print(F_torques.shape)

        F_term = np.vstack((F_forces, F_torques)).flatten()  # Combine forces and torques into 6x1 vector

        # === Get Disturbance ---
        # This is the external/unmodeled disturbance delta from the paper
        d = disturbance_func(t)

        # --- Dynamics Equation: Mx_dot + N = F + tau + d ---
        # Rearranging for x_dot: x_dot = M_inv * (F - N + tau + d)
        x_dot = self.M_inv @ (F_term - N_term + thrust_torque + d)

        # --- Combine state derivatives ---
        dXdt = np.concatenate((y_dot, x_dot))

        print(dXdt.shape)

        return dXdt

    def update_state(self, X_dot, dt):
        """Update state using Euler integration"""
        # NOTE: Using RK4 in simulation.py is preferred for accuracy.
        # This Euler update is kept for potential direct use but not used by main loop.
        self.X = self.X + X_dot * dt
        # Normalize angles if needed (e.g., psi to [-pi, pi])
        self.X[5] = (self.X[5] + np.pi) % (2 * np.pi) - np.pi  # Normalize Psi
        self.X[4] = np.clip(self.X[4], -np.pi / 2 + 0.01, np.pi / 2 - 0.01)  # Keep Theta away from singularity

    def get_state(self):
        """Get current state"""
        return self.X

    def get_pose(self):
        """Get current pose"""
        return self.X[0:6]  # zeta, gamma

    def get_velocity(self):
        """Get current velocity"""
        return self.X[6:12]  # v, omega


class AirshipCasADiSymbolic:
    """
    Airship symbolic model class
    """
    def __init__(self, input_params):
        self.params = input_params
        self.m = input_params.m
        self.g = input_params.g
        self.I0 = input_params.I0
        self.M = input_params.M_cfg
        self.M_inv = input_params.M_inv
        self.M_upper_left = self.M[0:3, 0:3]
        self.rc = input_params.rc.flatten()
        self.rb = input_params.rb.flatten()
        self.rp_r = input_params.rp_r.flatten()
        self.rp_l = input_params.rp_l.flatten()
        self.Vol_airship = input_params.Vol_airship
        self.rho_air = input_params.rho_air
        self.S_ref = input_params.S_ref
        self.L_ref = input_params.L_ref
        self.V_wind = input_params.V_WIND_ERF
        self.AERO_COEFFS = input_params.AERO_COEFFS

    def rhs_symbolic(self, X, U, t=None, external_disturbance=None):
        """
        Build symbolic RHS using CasADi.

        Args:
            X: 12x1 casadi SX state vector [zeta, gamma, v, omega]
            U: Control vector - either:
               - 3x1 [T, μ, v] for direct thrust control
               - 6x1 [T, μ, v, delta_RUDT, delta_RUDB, delta_ELVL, delta_ELVR] for full control
            t: Time (optional)
            external_disturbance: Optional external disturbance (6x1)

        Returns:
            dX/dt as casadi SX 12x1
        """
        # ca = __import__("casadi")  # dynamic import
        _ = t

        # === Deconstruct state ===
        _zeta = X[0:3]  # Position in ERF
        gamma = X[3:6]  # Attitude (Euler angles)
        v = X[6:9]  # Linear velocity in BRF
        omega = X[9:12]  # Angular velocity in BRF

        # === Kinematics ===
        R = R_block(gamma)  # Combined rotation matrix
        y_dot = R @ ca.vertcat(v, omega)  # [zeta_dot, gamma_dot]



        # === Dynamics ===

        #=================================================================
        #                     Coriolis and Centrifugal Effects
        #=================================================================
        # --- Calculate N term (Coriolis and centrifugal effects) ---
        omega_cross_v = ca.cross(omega, v)
        omega_cross_rc = ca.cross(omega, self.rc)
        omega_cross_omega_cross_rc = ca.cross(omega, omega_cross_rc)
        omega_cross_I0_omega = ca.cross(omega, self.I0 @ omega)
        rc_cross_omega_cross_v = ca.cross(self.rc, omega_cross_v)

        N1 = self.M_upper_left @ omega_cross_v + self.m * omega_cross_omega_cross_rc
        N2 = omega_cross_I0_omega + self.m * rc_cross_omega_cross_v
        N_term = ca.vertcat(N1, N2)

        # =============================================================
        #                     Gravity force and moment
        # =============================================================
        Rz = R_zeta(gamma)
        fg_earth = ca.vertcat(0, 0, self.m * self.g)  # Gravity in Earth Frame
        fg_BRF = Rz.T @ fg_earth  # Rotate gravity vector to Body Frame
        mg_BRF = ca.cross(self.rc, fg_BRF)  # Torque due to gravity acting at CG (rc is CV->CG)


        #========================================================================
        #                     Buoyancy force and moment
        #========================================================================
        F_buoy_earth = ca.vertcat(0, 0, -self.rho_air * self.Vol_airship * self.g)
        fb_BRF = Rz.T @ F_buoy_earth
        mb_BRF = ca.cross(-self.rb, fb_BRF)  # Torque due to buoyancy acting at CB (assumed at CV, so arm is -rb)


        #========================================================================
        #                     aerodynamic forces and moments
        #========================================================================
        # === Wind and relative velocity calculation ===
        V_wind_ERF = self.V_wind  # Wind velocity in ERF
        V_wind_BRF = Rz.T @ V_wind_ERF  # Transform wind to BRF

        # Calculate relative velocity
        v_rel_brf, u_rel, v_rel_body, w_rel = calculate_relative_velocity(v, V_wind_BRF)

        # Dynamic pressure
        V_rel_mag = ca.norm_2(v_rel_brf)
        q_dyn = 0.5 * self.rho_air * V_rel_mag**2

        # Angle of attack and sideslip angle
        alpha, beta = calculate_aoa_sideslip(u_rel, v_rel_body, w_rel, V_rel_mag, use_casadi=True)

        # === Get Control Surface Deflections ===
        # !!! Critical Placeholder: These values need to be determined by a Control Allocation module based on tau[3:6] !!!
        # !!! Critical Placeholder: These values need to be determined by a Control Allocation module based on tau[3:6] !!!
        delta_RUDT = np.deg2rad(0.0)  # [rad] - Placeholder
        delta_RUDB = np.deg2rad(0.0)  # [rad] - Placeholder
        delta_ELVL = np.deg2rad(0.0)  # [rad] - Placeholder
        delta_ELVR = np.deg2rad(0.0)  # [rad] - Placeholder

        # Use extracted function to calculate aerodynamic forces and moments
        fa_BRF, ma_BRF = calculate_aero_forces_moments(
            q_dyn, alpha, beta,
            self.AERO_COEFFS,
            delta_RUDT, delta_RUDB, delta_ELVL, delta_ELVR,
            use_casadi=True
        )



        #========================================================================
        #                     Thrust and torque
        #========================================================================
        # Check dimensions of U
        print(U.shape)
        # Directly convert thrust parameters to force and torque
        U_vec = thrust_params_to_force_torque(U, self.rp_r, self.rp_l, use_casadi=True)


        print(U_vec.shape)
        T_total = U_vec[0:3]  # Thrust vector
        tau_vec = U_vec[3:6]



        #==================================================================
        #                     Combine forces and moments
        #==================================================================
        F_forces = fg_BRF - fb_BRF + fa_BRF + T_total
        F_torques = mg_BRF + mb_BRF + ma_BRF + tau_vec
        F_term = ca.vertcat(F_forces, F_torques)

        # --- Add external disturbance if provided ---
        if external_disturbance is not None:
            F_term = F_term + external_disturbance

        # --- Dynamics equation: Mx_dot + N = F ---
        x_dot = self.M_inv @ (F_term - N_term)

        # --- Combine state derivatives ---
        dXdt = ca.vertcat(y_dot, x_dot)

        return dXdt

    def get_nmpc_model(self):
        """
        Create a CasADi function for use in NMPC.

        Returns:
            f: CasADi Function that maps (x, u) to xdot
        """
        # ca = __import__("casadi")

        # Define symbolic variables
        x = ca.SX.sym("x", 12)  # State
        u = ca.SX.sym("u", 3)  # Control (T, μ, ν)  , delta_RUDT, delta_RUDB, delta_ELVL, delta_ELVR)

        # Get dynamics
        xdot = self.rhs_symbolic(x, u)

        # Create function
        f = ca.Function("f", [x, u], [xdot], ["x", "u"], ["xdot"])

        return f

    def discrete_time_model(self, dt, integration_method="rk4"):
        """
        Get a discrete-time model using various integration methods.

        Args:
            dt: Time step
            integration_method: 'euler', 'rk4', etc.

        Returns:
            F: CasADi Function that maps (x, u) to x_next
        """
        # ca = __import__("casadi")

        # Define symbolic variables
        x = ca.SX.sym("x", 12)
        u = ca.SX.sym("u", 7)

        # Get continuous dynamics
        xdot = self.rhs_symbolic(x, u)

        # Create discrete model based on integration method
        if integration_method == "euler":
            x_next = x + dt * xdot
        elif integration_method == "rk4":
            # RK4 integration
            k1 = xdot
            k2 = self.rhs_symbolic(x + dt / 2 * k1, u)
            k3 = self.rhs_symbolic(x + dt / 2 * k2, u)
            k4 = self.rhs_symbolic(x + dt * k3, u)
            x_next = x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        else:
            raise ValueError(f"Unknown integration method: {integration_method}")

        # Create a function
        F = ca.Function("F", [x, u], [x_next], ["x", "u"], ["x_next"])

        return F


In [ ]:
"""
Trajectory generation module (trajectory.py)
"""
# pylint: disable=invalid-name
# pylint: disable=line-too-long
# cspell:ignore R_zeta R_y_inv Rc_z Rc_y_inv ddot arctan2 linalg xdot phiddot phidot    psiddot
# cspell:ignore phidot phiddot psidot psiddot thetaddot ydot

import numpy as np
from airship.utils import R_zeta, R_y_inv


class Trajectory:
    """
    Trajectory generation module


        Parameters:
            t: Current time
            start_point: Starting point coordinates [x, y, z], default is origin
            end_point: End point coordinates [x, y, z], default is [5000, 5000, -19000]
            speed: Flight speed in m/s
            hover_at_end: Whether to hover after reaching the end point, otherwise continue flying straight

        Returns:
            yc, yc_dot, yc_ddot, xc, xc_dot: Desired states and derivatives


        Description:
            - yc:

                Represents the desired state vector, containing the desired position and attitude of the airship.
                Specifically includes:
                    Position: [x, y, z], the desired position of the airship in space.
                    Attitude: [φ, θ, ψ], the desired attitude angles of the airship (roll, pitch, yaw).
            - yc_dot:

                Represents the first derivative of the desired state, i.e., the desired velocity vector.
                Specifically includes:
                    Linear velocity: [vx, vy, vz], the desired linear velocity of the airship in space.
                    Angular velocity: [ωφ, ωθ, ωψ], the desired angular velocity of the airship.
            - yc_ddot:

                Represents the second derivative of the desired state, i.e., the desired acceleration vector.
                Specifically includes:
                    Linear acceleration: [ax, ay, az], the desired linear acceleration of the airship in space.
                    Angular acceleration: [αφ, αθ, αψ], the desired angular acceleration of the airship.
            - xc:

                Represents the control command vector, containing desired linear and angular velocities.
                Specifically includes:
                    Linear velocity command: [vx, vy, vz], the desired linear velocity of the airship.
                    Angular velocity command: [ωφ, ωθ, ωψ], the desired angular velocity of the airship.
            - xc_dot:

                Represents the first derivative of the control command, i.e., the rate of change of control commands.
                Specifically includes:
                    Linear velocity rate: [dvx/dt, dvy/dt, dvz/dt], the time rate of change of linear velocity.
                    Angular velocity rate: [dωφ/dt, dωθ/dt, dωψ/dt], the time rate of change of angular velocity.
    """
    def __init__(self):
        pass  # No specific initialization needed for this trajectory

    # ┌─────────────────────────────────────────────────────┐
    # │          Spiral trajectory function                  │
    # └─────────────────────────────────────────────────────┘

    def get_spiral_trajectory(self, t):

        """
        Generate a spiral trajectory with altitude variation

        Args:
            t: Current time
        Returns:
            yc, yc_dot, yc_ddot, xc, xc_dot
        """

        dt_small = 1e-4

        # --- Trajectory parameters ---
        omega = 0.07  # Angular velocity
        r = 1500  # Radius
        h_max = 2000  # Maximum height

        # Print starting point information at initial time
        if abs(t) < 1e-3:  # When t approaches 0
            start_x = r * np.cos(0)  # = r = 1500
            start_y = r * np.sin(0)  # = 0
            start_z = h_max * (1 - np.exp(0))  # = 0
            print(f"[Spiral Trajectory] Starting point position: [{start_x:.1f}, {start_y:.1f}, {start_z:.1f}] (meters)")
            print(f"[Spiral Trajectory] Trajectory parameters: radius={r}m, max height={h_max}m, angular velocity={omega}rad/s")

        # --- Directly calculate position, velocity and acceleration ---
        theta = omega * t
        # Position
        xd = r * np.cos(theta)
        yd = r * np.sin(theta)
        zd = h_max * (1 - np.exp(-theta / 10))
        pos = np.array([xd, yd, zd])

        # Velocity
        xd_dot = -r * omega * np.sin(theta)
        yd_dot = r * omega * np.cos(theta)
        zd_dot = h_max * (1 / 10) * np.exp(-theta / 10) * omega
        vel = np.array([xd_dot, yd_dot, zd_dot])

        # Acceleration
        xd_ddot = -r * omega**2 * np.cos(theta)
        yd_ddot = -r * omega**2 * np.sin(theta)
        zd_ddot = -h_max * (1 / 10) * omega**2 * np.exp(-theta / 10)
        acc = np.array([xd_ddot, yd_ddot, zd_ddot])

        # Construct position and velocity vectors
        zeta_d = pos
        zeta_d_dot = vel
        zeta_d_ddot = acc

        # Calculate attitude
        phi_d = 0.0  # Maintain zero roll
        theta_d = np.arctan2(-vel[2], np.sqrt(vel[0] ** 2 + vel[1] ** 2))  # Pitch angle
        psi_d = np.arctan2(vel[1], vel[0])  # Yaw angle
        gamma_d = np.array([phi_d, theta_d, psi_d])

        # Use numerical differentiation to get attitude derivatives
        _, gamma_d_plus = self.get_spiral_pos_att(t + dt_small)
        _, gamma_d_minus = self.get_spiral_pos_att(t - dt_small)
        gamma_d_dot = (gamma_d_plus - gamma_d_minus) / (2 * dt_small)

        # Combine yc, yc_dot
        yc = np.concatenate((zeta_d, gamma_d))
        yc_dot = np.concatenate((zeta_d_dot, gamma_d_dot))

        # Velocity commands vc, wc
        Rc_z = R_zeta(gamma_d)
        Rc_y_inv = R_y_inv(gamma_d)
        vc = Rc_z.T @ zeta_d_dot.reshape(-1, 1)
        vc = vc.flatten()
        wc = Rc_y_inv @ gamma_d_dot.reshape(-1, 1)
        wc = wc.flatten()
        xc = np.concatenate((vc, wc))

        # xc_dot simplified approximation through symbolic derivatives
        vc_dot = Rc_z.T @ zeta_d_ddot.reshape(-1, 1)
        vc_dot = vc_dot.flatten()
        wc_dot = np.zeros(3)  # Simplified processing, assuming small angular velocity change rate
        xc_dot = np.concatenate((vc_dot, wc_dot))

        # yc_ddot simplified processing
        gamma_d_ddot = np.zeros(3)
        yc_ddot = np.concatenate((zeta_d_ddot, gamma_d_ddot))

        return yc, yc_dot, yc_ddot, xc, xc_dot

    def get_spiral_pos_att(self, t):
        """
        Calculate the position and attitude of the spiral trajectory at time t, used for derivative calculation
        Avoid recursive calls to define_spiral_trajectory
        """
        # --- Trajectory parameters ---
        omega = 0.07  # Angular velocity (rad/s)
        r = 1500  # Basic radius (m)
        h_max = 2000  # Maximum height (m)

        # --- Position calculation ---
        theta = omega * t
        xd = r * np.cos(theta)
        yd = r * np.sin(theta)
        zd = h_max * (1 - np.exp(-theta / 10))
        zeta_d = np.array([xd, yd, zd])

        # --- Velocity calculation (for attitude determination) ---
        xd_dot = -r * omega * np.sin(theta)
        yd_dot = r * omega * np.cos(theta)
        zd_dot = h_max * (1 / 10) * np.exp(-theta / 10) * omega

        # --- Attitude calculation ---
        phi_d = 0.0  # Maintain zero roll
        theta_d = np.arctan2(-zd_dot, np.sqrt(xd_dot**2 + yd_dot**2))  # Pitch angle
        psi_d = np.arctan2(yd_dot, xd_dot)  # Yaw angle
        gamma_d = np.array([phi_d, theta_d, psi_d])

        return zeta_d, gamma_d

    # ********************* Figure-8 trajectory function *********************

    def get_figure8_trajectory(self, t):
        """
        Generate a horizontal figure-8 trajectory with smooth altitude variation
        Return the desired state and derivatives of the figure-8 trajectory
        Parameters:
            t: Current time

        Returns:
            yc, yc_dot, yc_ddot, xc, xc_dot: Same output format as get_desired_state
        """
        dt_small = 1e-4

        # --- Trajectory parameters ---
        a = 3000  # Width of figure-8
        b = 2000  # Height of figure-8
        omega = 0.003  # Angular velocity, controls the speed of movement along the trajectory
        h_center = -19000  # Center altitude
        h_amp = 500  # Altitude oscillation amplitude
        omega_h = 0.002  # Angular velocity of altitude variation

        # Print starting point information at initial time
        if abs(t) < 1e-3:  # When t approaches 0
            start_x = a * np.sin(0)  # = 0
            start_y = b * np.sin(0) * np.cos(0)  # = 0
            start_z = h_center + h_amp * np.sin(0)  # = h_center = -19000
            print(f"[Figure-8 Trajectory] Starting point position: [{start_x:.1f}, {start_y:.1f}, {start_z:.1f}] (meters)")
            print(f"[Figure-8 Trajectory] Trajectory parameters: width={a}m, height={b}m, center altitude={h_center}m, angular velocity={omega}rad/s")

        # --- Position calculation ---
        # Parametric equations for figure-8
        xd = a * np.sin(omega * t)
        yd = b * np.sin(omega * t) * np.cos(omega * t)
        zd = h_center + h_amp * np.sin(omega_h * t)
        zeta_d = np.array([xd, yd, zd])

        # --- Velocity calculation ---
        xd_dot = a * omega * np.cos(omega * t)
        yd_dot = b * omega * (np.cos(omega * t) * np.cos(omega * t)
                              - np.sin(omega * t) * np.sin(omega * t)
                              )
        zd_dot = h_amp * omega_h * np.cos(omega_h * t)
        zeta_d_dot = np.array([xd_dot, yd_dot, zd_dot])

        # --- Acceleration calculation (using numerical differentiation) ---
        # Calculate velocity at t+dt time
        xd_dot_plus = a * omega * np.cos(omega * (t + dt_small))
        yd_dot_plus = (
            b * omega * (np.cos(omega * (t + dt_small)) * np.cos(omega * (t + dt_small))
                         - np.sin(omega * (t + dt_small)) * np.sin(omega * (t + dt_small))
                         )
        )
        zd_dot_plus = h_amp * omega_h * np.cos(omega_h * (t + dt_small))

        # Calculate velocity at t-dt time
        xd_dot_minus = a * omega * np.cos(omega * (t - dt_small))
        yd_dot_minus = (
            b * omega * (np.cos(omega * (t - dt_small)) * np.cos(omega * (t - dt_small))
               - np.sin(omega * (t - dt_small)) * np.sin(omega * (t - dt_small)))
        )
        zd_dot_minus = h_amp * omega_h * np.cos(omega_h * (t - dt_small))

        # Calculate acceleration using central difference
        zeta_d_ddot = np.array(
            [
                (xd_dot_plus - xd_dot_minus) / (2 * dt_small),
                (yd_dot_plus - yd_dot_minus) / (2 * dt_small),
                (zd_dot_plus - zd_dot_minus) / (2 * dt_small),
            ]
        )

        # --- Attitude calculation ---
        # Calculate desired heading angle (tangent direction)
        phi_d = 0.0  # Maintain zero roll
        theta_d = np.arctan2(-zd_dot, np.sqrt(xd_dot**2 + yd_dot**2))  # Pitch angle
        psi_d = np.arctan2(yd_dot, xd_dot)  # Yaw angle
        gamma_d = np.array([phi_d, theta_d, psi_d])

        # --- Attitude derivative calculation ---
        # Use numerical differentiation to obtain attitude derivatives
        _, gamma_d_plus = self.get_figure8_pos_att(t + dt_small)
        _, gamma_d_minus = self.get_figure8_pos_att(t - dt_small)
        gamma_d_dot = (gamma_d_plus - gamma_d_minus) / (2 * dt_small)

        # --- Combine yc, yc_dot ---
        yc = np.concatenate((zeta_d, gamma_d))
        yc_dot = np.concatenate((zeta_d_dot, gamma_d_dot))

        # --- Velocity commands vc, wc ---
        Rc_z = R_zeta(gamma_d)
        Rc_y_inv = R_y_inv(gamma_d)
        vc = Rc_z.T @ zeta_d_dot.reshape(-1, 1)
        vc = vc.flatten()
        wc = Rc_y_inv @ gamma_d_dot.reshape(-1, 1)
        wc = wc.flatten()
        xc = np.concatenate((vc, wc))

        # --- xc_dot simplified approximation using symbolic derivatives ---
        vc_dot = Rc_z.T @ zeta_d_ddot.reshape(-1, 1)
        vc_dot = vc_dot.flatten()
        wc_dot = np.zeros(3)  # Simplified processing, assuming small angular velocity rate changes
        xc_dot = np.concatenate((vc_dot, wc_dot))

        # --- yc_ddot also simplified processing ---
        gamma_d_ddot = np.zeros(3)
        yc_ddot = np.concatenate((zeta_d_ddot, gamma_d_ddot))

        return yc, yc_dot, yc_ddot, xc, xc_dot

    def get_figure8_pos_att(self, t):
        """
        Only calculate the position and attitude of the figure-8 trajectory at time t for derivative calculation
        Avoid recursive calls to get_figure8_trajectory
        """
        # --- Trajectory parameters ---
        a = 3000  # Width of the figure-8
        b = 2000  # Height of the figure-8
        omega = 0.003  # Angular velocity, controls airship movement speed on trajectory
        h_center = -19000  # Center altitude
        h_amp = 500  # Altitude oscillation amplitude
        omega_h = 0.002  # Angular velocity for altitude variation

        # --- Position ---
        xd = a * np.sin(omega * t)
        yd = b * np.sin(omega * t) * np.cos(omega * t)
        zd = h_center + h_amp * np.sin(omega_h * t)
        zeta_d = np.array([xd, yd, zd])

        # --- Velocity (for attitude calculation) ---
        xd_dot = a * omega * np.cos(omega * t)
        yd_dot = b * omega * (np.cos(omega * t) * np.cos(omega * t)
                              - np.sin(omega * t) * np.sin(omega * t)
                              )
        zd_dot = h_amp * omega_h * np.cos(omega_h * t)

        # --- Attitude ---
        phi_d = 0.0
        theta_d = np.arctan2(-zd_dot, np.sqrt(xd_dot**2 + yd_dot**2))
        psi_d = np.arctan2(yd_dot, xd_dot)
        gamma_d = np.array([phi_d, theta_d, psi_d])

        return zeta_d, gamma_d

    # ********************* Lemniscate trajectory function *********************

    def get_lemniscate_trajectory(self, t):
        """
        Generate Lemniscate trajectory, resembling infinity symbol, with altitude variation

        Parameters:
            t: Current time

        Returns:
            yc, yc_dot, yc_ddot, xc, xc_dot: Desired states and their derivatives
        """
        dt_small = 1e-4

        # --- Trajectory parameters ---
        a = 2500  # Curve scale parameter
        omega = 0.004  # Angular velocity
        h_center = -19000  # Center altitude
        h_amp = 800  # Altitude variation amplitude
        h_freq = 0.001  # Altitude variation frequency

        # Print starting point information at initial time
        if abs(t) < 1e-3:  # When t is close to 0
            theta = 0
            denom = 1 + np.sin(theta) ** 2  # = 1
            start_x = a * np.cos(theta) / denom  # = a = 2500
            start_y = a * np.sin(theta) * np.cos(theta) / denom  # = 0
            start_z = h_center + h_amp * np.sin(0)  # = h_center = -19000
            print(f"[Lemniscate Trajectory] Starting point position: [{start_x:.1f}, {start_y:.1f}, {start_z:.1f}] (meters)")
            print(f"[Lemniscate Trajectory] Trajectory parameters: scale={a}m, center altitude={h_center}m, angular velocity={omega}rad/s")

        # --- Parametric curve parameters ---
        theta = omega * t
        # Lemniscate parametric equations
        denom = 1 + np.sin(theta) ** 2
        xd = a * np.cos(theta) / denom
        yd = a * np.sin(theta) * np.cos(theta) / denom
        zd = h_center + h_amp * np.sin(h_freq * t)
        zeta_d = np.array([xd, yd, zd])

        # --- Velocity calculation (analytical derivatives) ---
        xd_dot_num = (
            -a * np.sin(theta) * denom
            - a * np.cos(theta) * 2 * np.sin(theta) * np.cos(theta)
        )
        yd_dot_num = (
            a * (np.cos(theta) ** 2 - np.sin(theta) ** 2) * denom
            - a * np.sin(theta) * np.cos(theta) * 2 * np.sin(theta) * np.cos(theta)
        )
        xd_dot = (xd_dot_num / denom**2) * omega
        yd_dot = (yd_dot_num / denom**2) * omega
        zd_dot = h_amp * h_freq * np.cos(h_freq * t)
        zeta_d_dot = np.array([xd_dot, yd_dot, zd_dot])

        # --- Calculate acceleration using numerical differentiation ---
        # Calculate position and velocity at t+dt time
        theta_plus = omega * (t + dt_small)
        denom_plus = 1 + np.sin(theta_plus) ** 2

        xd_dot_num_plus = (
            -a * np.sin(theta_plus) * denom_plus
            - a * np.cos(theta_plus) * 2 * np.sin(theta_plus) * np.cos(theta_plus)
        )
        yd_dot_num_plus = (
            a * (np.cos(theta_plus) ** 2 - np.sin(theta_plus) ** 2) * denom_plus
            - a * np.sin(theta_plus)
            * np.cos(theta_plus)
            * 2
            * np.sin(theta_plus)
            * np.cos(theta_plus)
        )

        xd_dot_plus = (xd_dot_num_plus / denom_plus**2) * omega
        yd_dot_plus = (yd_dot_num_plus / denom_plus**2) * omega
        zd_dot_plus = h_amp * h_freq * np.cos(h_freq * (t + dt_small))

        # Calculate position and velocity at t-dt time
        theta_minus = omega * (t - dt_small)
        denom_minus = 1 + np.sin(theta_minus) ** 2

        xd_dot_num_minus = (
            -a * np.sin(theta_minus) * denom_minus
            - a * np.cos(theta_minus) * 2 * np.sin(theta_minus) * np.cos(theta_minus)
        )
        yd_dot_num_minus = (
            a * (np.cos(theta_minus) ** 2
            - np.sin(theta_minus) ** 2) * denom_minus
            - (a * np.sin(theta_minus) * np.cos(theta_minus)
                * 2 * np.sin(theta_minus) * np.cos(theta_minus))
        )
        xd_dot_minus = (xd_dot_num_minus / denom_minus**2) * omega
        yd_dot_minus = (yd_dot_num_minus / denom_minus**2) * omega
        zd_dot_minus = h_amp * h_freq * np.cos(h_freq * (t - dt_small))

        # Calculate acceleration using central difference
        zeta_d_ddot = np.array(
            [
                (xd_dot_plus - xd_dot_minus) / (2 * dt_small),
                (yd_dot_plus - yd_dot_minus) / (2 * dt_small),
                (zd_dot_plus - zd_dot_minus) / (2 * dt_small),
            ]
        )

        # --- Attitude calculation ---
        phi_d = 0.0  # Maintain zero roll
        theta_d = np.arctan2(-zd_dot, np.sqrt(xd_dot**2 + yd_dot**2))  # Pitch angle
        psi_d = np.arctan2(yd_dot, xd_dot)  # Yaw angle
        gamma_d = np.array([phi_d, theta_d, psi_d])

        # --- Attitude derivative calculation (using auxiliary function) ---
        _, gamma_d_plus = self.get_lemniscate_pos_att(t + dt_small)
        _, gamma_d_minus = self.get_lemniscate_pos_att(t - dt_small)
        gamma_d_dot = (gamma_d_plus - gamma_d_minus) / (2 * dt_small)

        # --- Combine yc, yc_dot ---
        yc = np.concatenate((zeta_d, gamma_d))
        yc_dot = np.concatenate((zeta_d_dot, gamma_d_dot))

        # --- Velocity commands vc, wc ---
        Rc_z = R_zeta(gamma_d)
        Rc_y_inv = R_y_inv(gamma_d)
        vc = Rc_z.T @ zeta_d_dot.reshape(-1, 1)
        vc = vc.flatten()
        wc = Rc_y_inv @ gamma_d_dot.reshape(-1, 1)
        wc = wc.flatten()
        xc = np.concatenate((vc, wc))

        # --- xc_dot and yc_ddot ---
        vc_dot = Rc_z.T @ zeta_d_ddot.reshape(-1, 1)
        vc_dot = vc_dot.flatten()
        wc_dot = np.zeros(3)  # Simplified processing
        xc_dot = np.concatenate((vc_dot, wc_dot))

        gamma_d_ddot = np.zeros(3)
        yc_ddot = np.concatenate((zeta_d_ddot, gamma_d_ddot))

        return yc, yc_dot, yc_ddot, xc, xc_dot

    def get_lemniscate_pos_att(self, t):
        """Calculate lemniscate position and attitude at time t for derivative computation"""
        # --- Trajectory parameters ---
        a = 2500
        omega = 0.004
        h_center = -19000
        h_amp = 800
        h_freq = 0.001

        # --- Parametric curve parameters ---
        theta = omega * t
        # Lemniscate parametric equations
        denom = 1 + np.sin(theta) ** 2
        xd = a * np.cos(theta) / denom
        yd = a * np.sin(theta) * np.cos(theta) / denom
        zd = h_center + h_amp * np.sin(h_freq * t)
        zeta_d = np.array([xd, yd, zd])

        # --- Velocity calculation ---
        xd_dot_num = (
            -a * np.sin(theta) * denom
            - a * np.cos(theta) * 2 * np.sin(theta) * np.cos(theta)
        )
        yd_dot_num = (
            a * (np.cos(theta) ** 2 - np.sin(theta) ** 2) * denom
            - a * np.sin(theta) * np.cos(theta) * 2 * np.sin(theta) * np.cos(theta)
        )
        xd_dot = (xd_dot_num / denom**2) * omega
        yd_dot = (yd_dot_num / denom**2) * omega
        zd_dot = h_amp * h_freq * np.cos(h_freq * t)

        # --- Attitude ---
        phi_d = 0.0  # Maintain zero roll
        theta_d = np.arctan2(-zd_dot, np.sqrt(xd_dot**2 + yd_dot**2))
        psi_d = np.arctan2(yd_dot, xd_dot)
        gamma_d = np.array([phi_d, theta_d, psi_d])

        return zeta_d, gamma_d

    # ********************* Straight line trajectory function *********************
    def get_linear_trajectory(self, t, start_point=None, end_point=None, speed=10.0, hover_at_end=True):
        """
        Generate a straight line trajectory from start point to end point

        Parameters:
            t: Current time
            start_point: Starting coordinates [x, y, z], defaults to origin
            end_point: Ending coordinates [x, y, z], defaults to [5000, 5000, -19000]
            speed: Flight speed in m/s
            hover_at_end: Whether to hover at end point, otherwise continue flying straight

        Returns:
            yc, yc_dot, yc_ddot, xc, xc_dot: Desired states and derivatives
        Notes:
            - yc:
                Represents the desired state vector, containing airship's desired position and attitude.
                Specifically includes:
                    Position: [x, y, z], airship's desired position in space.
                    Attitude: [φ, θ, ψ], airship's desired attitude angles (roll, pitch, yaw).
            - yc_dot:
                Represents the first derivative of desired state, i.e., desired velocity vector.
                Specifically includes:
                    Linear velocity: [vx, vy, vz], airship's desired linear velocity in space.
                    Angular velocity: [ωφ, ωθ, ωψ], airship's desired angular velocity.
            - yc_ddot:
                Represents the second derivative of desired state, i.e., desired acceleration vector.
                Specifically includes:
                    Linear acceleration: [ax, ay, az], airship's desired linear acceleration in space.
                    Angular acceleration: [αφ, αθ, αψ], airship's desired angular acceleration.
            - xc:
                Represents the control command vector, containing desired linear and angular velocities.
                Specifically includes:
                    Linear velocity command: [vx, vy, vz], airship's desired linear velocity.
                    Angular velocity command: [ωφ, ωθ, ωψ], airship's desired angular velocity.
            - xc_dot:
                Represents the first derivative of control commands, i.e., rate of change of control commands.
                Specifically includes:
                    Linear velocity rate: [dvx/dt, dvy/dt, dvz/dt], time rate of change of linear velocity.
                    Angular velocity rate: [dωφ/dt, dωθ/dt, dωψ/dt], time rate of change of angular velocity.
        """
        dt_small = 1e-4

        # --- Trajectory parameters ---
        if start_point is None:
            start_point = np.array([0.0, 0.0, -19000.0])  # Default start point
        if end_point is None:
            end_point = np.array([5000.0, 5000.0, -19000.0])  # Default end point

        # Calculate direction vector and distance
        direction = end_point - start_point
        distance = np.linalg.norm(direction)
        unit_direction = direction / max(distance, 1e-10)  # Avoid division by zero

        # Calculate total flight time
        total_time = distance / speed

        # --- Position calculation ---
        if t < total_time or not hover_at_end:
            # Still flying, or no need to hover
            effective_t = t if hover_at_end else t % total_time
            progress = min(effective_t / total_time, 1.0) if hover_at_end else effective_t / total_time

            # Linear interpolation to calculate current position
            zeta_d = start_point + progress * direction
        else:
            # Reached end point and need to hover
            zeta_d = end_point

        # --- Velocity calculation ---
        if t < total_time or not hover_at_end:
            # Constant velocity during flight
            if not hover_at_end or t < total_time:
                zeta_d_dot = unit_direction * speed
            else:
                # Zero velocity after reaching end point
                zeta_d_dot = np.zeros(3)
        else:
            # Hovering state, zero velocity
            zeta_d_dot = np.zeros(3)

        # --- Acceleration calculation (theoretically zero, but kept for compatibility) ---
        zeta_d_ddot = np.zeros(3)

        # --- Attitude calculation ---
        # Calculate desired heading angle (towards forward direction)
        if np.linalg.norm(zeta_d_dot) > 1e-6:  # If there is velocity
            phi_d = 0.0  # Maintain zero roll
            theta_d = np.arctan2(-zeta_d_dot[2], np.sqrt(zeta_d_dot[0] ** 2 + zeta_d_dot[1] ** 2))  # Pitch angle
            psi_d = np.arctan2(zeta_d_dot[1], zeta_d_dot[0])  # Yaw angle
        else:
            # Hovering state maintains last attitude
            phi_d, theta_d, psi_d = self.get_linear_pos_att(max(0, t - dt_small), start_point, end_point, speed, hover_at_end)[1]

        gamma_d = np.array([phi_d, theta_d, psi_d])

        # --- Attitude derivative calculation ---
        # Use numerical differentiation to obtain attitude derivatives
        _, gamma_d_plus = self.get_linear_pos_att(t + dt_small, start_point, end_point, speed, hover_at_end)
        _, gamma_d_minus = self.get_linear_pos_att(t - dt_small, start_point, end_point, speed, hover_at_end)
        gamma_d_dot = (gamma_d_plus - gamma_d_minus) / (2 * dt_small)

        # --- Combine yc, yc_dot ---
        yc = np.concatenate((zeta_d, gamma_d)) # Position and attitude
        yc_dot = np.concatenate((zeta_d_dot, gamma_d_dot)) # Linear velocity and attitude angular velocity

        # --- Velocity commands vc, wc ---
        Rc_z = R_zeta(gamma_d)
        Rc_y_inv = R_y_inv(gamma_d)
        vc = Rc_z.T @ zeta_d_dot.reshape(-1, 1)
        vc = vc.flatten()
        wc = Rc_y_inv @ gamma_d_dot.reshape(-1, 1)
        wc = wc.flatten()
        xc = np.concatenate((vc, wc)) # Contains desired linear and angular velocities (control command vector)

        # --- xc_dot simplified approximation using symbolic derivatives ---
        vc_dot = Rc_z.T @ zeta_d_ddot.reshape(-1, 1)
        vc_dot = vc_dot.flatten()
        wc_dot = np.zeros(3)  #
        xc_dot = np.concatenate((vc_dot, wc_dot)) # First derivative of control commands, i.e., rate of change of control commands, linear velocity rate and angular velocity rate

        # --- yc_ddot also simplified processing ---
        gamma_d_ddot = np.zeros(3)
        yc_ddot = np.concatenate((zeta_d_ddot, gamma_d_ddot)) # Second derivative of desired state, i.e., desired linear acceleration and angular acceleration vector.

        return yc, yc_dot, yc_ddot, xc, xc_dot

    def get_linear_pos_att(self, t, start_point, end_point, speed, hover_at_end):
        """
        Calculate linear trajectory position and attitude at time t for derivative computation
        Avoid recursive calls to get_linear_trajectory
        args:
            t: Current time
            start_point: Starting coordinates [x, y, z]
            end_point: Ending coordinates [x, y, z]
            speed: Flight speed in m/s
            hover_at_end: Whether to hover at end point, otherwise continue flying straight
        return:
            zeta_d: Desired position
            gamma_d: Desired attitude
        """
        # Calculate direction vector and distance
        direction = end_point - start_point
        distance = np.linalg.norm(direction)
        unit_direction = direction / max(distance, 1e-10)  # Avoid division by zero

        # Calculate total flight time
        total_time = distance / speed

        # --- Position calculation ---
        if t < total_time or not hover_at_end:
            effective_t = t if hover_at_end else t % total_time
            progress = min(effective_t / total_time, 1.0) if hover_at_end else effective_t / total_time

            # Linear interpolation to calculate current position
            zeta_d = start_point + progress * direction
        else:
            # Reached end point and need to hover
            zeta_d = end_point

        # --- Velocity calculation (for attitude calculation) ---
        if t < total_time or not hover_at_end:
            if not hover_at_end or t < total_time:
                zeta_d_dot = unit_direction * speed
            else:
                zeta_d_dot = np.zeros(3)
        else:
            zeta_d_dot = np.zeros(3)

        # --- Attitude calculation ---
        if np.linalg.norm(zeta_d_dot) > 1e-6:  # If there is velocity
            phi_d = 0.0  # Maintain zero roll
            theta_d = np.arctan2(-zeta_d_dot[2],
                                 np.sqrt(zeta_d_dot[0] ** 2 + zeta_d_dot[1] ** 2))  # Pitch angle
            psi_d = np.arctan2(zeta_d_dot[1], zeta_d_dot[0])  # Yaw angle
        else:
            # Hovering state maintains last attitude, simplified here as default attitude
            phi_d = 0.0
            theta_d = 0.0
            # Use direction vector to calculate default heading angle
            if np.linalg.norm(direction[:2]) > 1e-6:
                psi_d = np.arctan2(direction[1], direction[0])
            else:
                psi_d = 0.0  # Default heading angle

        gamma_d = np.array([phi_d, theta_d, psi_d])

        return zeta_d, gamma_d


In [ ]:
"""
NMPC Controller Implementation based on do-mpc
"""

# pylint: disable=invalid-name
# cspell:ignore dompc vertcat radau mterm lterm rterm ndarray fmin fmax idas abstol reltol
# cspell: ignore nlpsol ipopt print_level max_iter acceptable_tol acceptable_obj_change_tol tol
# cspell: ignore cvodes mu_strategy hessian_approximation limited_memory_max_history alpha_for_y recalc_y max_wall_time print_time

# standard library
import logging

# third-party library
import numpy as np
import casadi as ca
import do_mpc

# local module
from config import parameters as params
from AirshipModeling.airship_dynamic import AirshipCasADiSymbolic
from AirshipModeling.thrust_vectoring import thrust_params_to_force_torque
from AirshipModeling.observer import DisturbanceObserver

# set up logger
logger = logging.getLogger(__name__)



class do_mpc_controller:
    """
    Airship NMPC Controller based on do-mpc
    """

    def __init__(self, use_disturbance_compensation=True, create_simulator=True):
        """
        Initialize do-mpc based controller

        Args:
            use_disturbance_compensation: Whether to enable disturbance compensation
            create_simulator: Whether to create do-mpc Simulator
        """
        self.use_disturbance_compensation = use_disturbance_compensation
        self.params = params
        self.create_simulator = create_simulator

        # Initialize disturbance observer
        if use_disturbance_compensation:
            self.disturbance_compensation_factor = getattr(params, 'do_compensation_gain', 0.9)
            self.last_disturbance_estimate = np.zeros(6)
            self.disturbance_observer = DisturbanceObserver()
        else:
            self.disturbance_observer = None

        # Create do-mpc model
        self.model = self._create_model()

        # Create MPC controller
        self.mpc = self._create_mpc_controller()

        # Create estimator
        self.estimator = self._create_estimator()

        # Create Simulator (if needed)
        if create_simulator:
            self.simulator = self._create_simulator()
        else:
            self.simulator = None

        # Initialize controller
        self._setup_initial_conditions()

        # Store last control input
        self.last_control = np.array([5.0, 0.0, 0.0])

    def _create_model(self):
        """
        Create do-mpc model

        Returns:
            do_mpc.model.Model: Airship dynamics model
        """
        # Create model type (continuous time)
        model_type = 'continuous'
        model = do_mpc.model.Model(model_type)

        # Define state variables - 12-dimensional state vector
        pos = model.set_variable(var_type='_x', var_name='pos', shape=(3, 1))      # Position [x, y, z]
        att = model.set_variable(var_type='_x', var_name='att', shape=(3, 1))      # Attitude [phi, theta, psi]
        vel = model.set_variable(var_type='_x', var_name='vel', shape=(3, 1))      # Linear velocity [u, v, w]
        omega = model.set_variable(var_type='_x', var_name='omega', shape=(3, 1))  # Angular velocity [p, q, r]

        # Define control inputs - 3-dimensional control vector
        T = model.set_variable(var_type='_u', var_name='T')          # Thrust magnitude
        mu = model.set_variable(var_type='_u', var_name='mu')        # Horizontal deflection angle
        nu = model.set_variable(var_type='_u', var_name='nu')        # Vertical deflection angle

        # Define reference trajectory parameters
        pos_ref = model.set_variable(var_type='_p', var_name='pos_ref', shape=(3, 1))
        att_ref = model.set_variable(var_type='_p', var_name='att_ref', shape=(3, 1))
        vel_ref = model.set_variable(var_type='_p', var_name='vel_ref', shape=(3, 1))
        omega_ref = model.set_variable(var_type='_p', var_name='omega_ref', shape=(3, 1))

        # Define disturbance variables (for Simulator)
        disturbance = model.set_variable(var_type='_p', var_name='disturbance', shape=(6, 1))


        # Use existing symbolic dynamics model
        symbolic_model = AirshipCasADiSymbolic(self.params)

        # Combine state vector
        X_state = ca.vertcat(pos, att, vel, omega)
        U_control = ca.vertcat(T, mu, nu)

        # Get dynamics equations (including disturbance)
        X_dot = symbolic_model.rhs_symbolic(X_state, U_control, external_disturbance=disturbance)

        # Add numerical stability - limit derivative magnitudes
        max_derivative = 1e5
        X_dot = ca.fmin(ca.fmax(X_dot, -max_derivative), max_derivative)

        # Decompose state derivatives
        pos_dot = X_dot[0:3]
        att_dot = X_dot[3:6]
        vel_dot = X_dot[6:9]
        omega_dot = X_dot[9:12]

        # Set differential equations
        model.set_rhs('pos', pos_dot)
        model.set_rhs('att', att_dot)
        model.set_rhs('vel', vel_dot)
        model.set_rhs('omega', omega_dot)

        # Set state expressions (for objective function)
        position_error = pos - pos_ref
        attitude_error = att - att_ref
        velocity_error = vel - vel_ref
        angular_error = omega - omega_ref

        # Auxiliary expressions
        model.set_expression('pos_error', position_error)
        model.set_expression('att_error', attitude_error)
        model.set_expression('vel_error', velocity_error)
        model.set_expression('ang_error', angular_error)

        # Add measurement output (for estimator)
        model.set_expression('y_meas', X_state)  # Assume full state observability

        # Complete model setup
        model.setup()

        return model

    def _create_mpc_controller(self):
        """
        Create MPC controller - Enhanced version

        Returns:
            do_mpc.controller.MPC: Configured MPC controller
        """
        mpc = do_mpc.controller.MPC(self.model)

        # MPC setup
        setup_mpc = {
            'n_horizon': min(params.N_HORIZON, 8),
            'n_robust': 1,
            'open_loop': 0,
            't_step': params.DT,
            'state_discretization': 'collocation',  # Back to collocation
            'collocation_type': 'radau',  # Change to legendre for better stability
            'collocation_deg': 2,
            'collocation_ni': 1,  # Reduce internal point count
            'store_full_solution': False, #True,
            # Optimized solver options
            'nlpsol_opts': {
                'ipopt.print_level': 0,
                'ipopt.max_iter': 50,  # Reduce maximum iterations
                'ipopt.acceptable_tol': 1e-3,  # Relax tolerance
                'ipopt.acceptable_obj_change_tol': 1e-3,
                'ipopt.tol': 1e-3,
                'ipopt.mu_strategy': 'adaptive',
                'ipopt.hessian_approximation': 'limited-memory',
                'ipopt.limited_memory_max_history': 5,  # Reduce history records
                'ipopt.alpha_for_y': 'primal',
                'ipopt.recalc_y': 'yes',
                'ipopt.max_wall_time': 3.0,  # Further limit solving time
                'ipopt.warm_start_init_point': 'yes',  # Enable warm start
                'print_time': 0
            }
        }

        mpc.set_param(**setup_mpc)

        # Set uncertainty values (disturbance compensation)
        mpc.set_uncertainty_values(
            pos_ref=np.zeros((3, 1)),
            att_ref=np.zeros((3, 1)),
            vel_ref=np.zeros((3, 1)),
            omega_ref=np.zeros((3, 1)),
            disturbance=np.zeros((6, 1))  # Add disturbance parameters
        )

        # Improved objective function - numerical scaling
        # Scale weight matrices to avoid numerical issues
        Q_scaled = params.Q * 0.01  # Reduce state weights
        Qf_scaled = params.Qf * 0.01
        R_scaled = params.R * 100.0  # Increase control weights to promote smoothness



        # Terminal cost - only includes state errors, not control inputs
        mterm = (self.model.aux['pos_error'].T @ Qf_scaled[0:3, 0:3] @ self.model.aux['pos_error'] +
                 self.model.aux['att_error'].T @ Qf_scaled[3:6, 3:6] @ self.model.aux['att_error'] +
                 self.model.aux['vel_error'].T @ Qf_scaled[6:9, 6:9] @ self.model.aux['vel_error'] +
                 self.model.aux['ang_error'].T @ Qf_scaled[9:12, 9:12] @ self.model.aux['ang_error'])

        # Stage cost - includes state errors and control inputs
        lterm = (self.model.aux['pos_error'].T @ Q_scaled[0:3, 0:3] @ self.model.aux['pos_error'] +
                 self.model.aux['att_error'].T @ Q_scaled[3:6, 3:6] @ self.model.aux['att_error'] +
                 self.model.aux['vel_error'].T @ Q_scaled[6:9, 6:9] @ self.model.aux['vel_error'] +
                 self.model.aux['ang_error'].T @ Q_scaled[9:12, 9:12] @ self.model.aux['ang_error'] +
                 self.model.u['T']**2 * R_scaled[0, 0] +
                 self.model.u['mu']**2 * R_scaled[1, 1] +
                 self.model.u['nu']**2 * R_scaled[2, 2])

        mpc.set_objective(mterm=mterm, lterm=lterm)

        # Control input regularization
        mpc.set_rterm(T=0.1, mu=0.1, nu=0.1)

        # Set constraints
        self._set_mpc_constraints(mpc)

        # Complete MPC setup
        mpc.setup()

        return mpc

    def _set_mpc_constraints(self, mpc):
        """Set MPC constraints"""
        # Control input constraints
        mpc.bounds['lower', '_u', 'T'] = max(params.T_MIN, 0.1)  # Avoid zero thrust
        mpc.bounds['upper', '_u', 'T'] = params.T_MAX
        mpc.bounds['lower', '_u', 'mu'] = params.MU_MIN
        mpc.bounds['upper', '_u', 'mu'] = params.MU_MAX
        mpc.bounds['lower', '_u', 'nu'] = params.NU_MIN
        mpc.bounds['upper', '_u', 'nu'] = params.NU_MAX

        # State constraints
        max_position = 200.0  # Reduce to reasonable range
        max_angle = np.pi/2   # Limit attitude angles to prevent singularities
        max_velocity = 30.0
        max_angular_velocity = np.pi/2  # Reasonable angular velocity limits

        # Position constraints
        mpc.bounds['lower', '_x', 'pos'] = -max_position
        mpc.bounds['upper', '_x', 'pos'] = max_position

        # Attitude constraints
        mpc.bounds['lower', '_x', 'att'] = -max_angle
        mpc.bounds['upper', '_x', 'att'] = max_angle
        mpc.bounds['lower', '_x', 'att', 1] = -max_angle/6  # Pitch angle theta limits
        mpc.bounds['upper', '_x', 'att', 1] = max_angle/6

        # Velocity constraints
        mpc.bounds['lower', '_x', 'vel'] = -max_velocity
        mpc.bounds['upper', '_x', 'vel'] = max_velocity
        mpc.bounds['lower', '_x', 'omega'] = -max_angular_velocity
        mpc.bounds['upper', '_x', 'omega'] = max_angular_velocity

    def _create_estimator(self):
        """
        Create state estimator

        Returns:
                State feedback estimator
        """
        estimator = do_mpc.estimator.StateFeedback(self.model)
        return estimator

    def _create_simulator(self):
        """
        Create do-mpc Simulator - Enhanced version for standalone use

        Returns:
            do_mpc.simulator.Simulator: Configured simulator
        """
        if not self.create_simulator:
            return None

        simulator = do_mpc.simulator.Simulator(self.model)

        # Enhanced simulator setup for stability
        setup_simulator = {
            't_step': params.DT,
            'integration_tool': 'cvodes',  # Use CVODES for better stability
            'abstol': 1e-8,  # Tighter absolute tolerance
            'reltol': 1e-6,  # Tighter relative tolerance
            'max_step_size': params.DT / 10,  # Limit maximum step size
            'min_step_size': params.DT / 1000,  # Set minimum step size
        }

        simulator.set_param(**setup_simulator)

        # Enhanced parameter function with error handling
        def disturbance_func(t_now):
            """Define time-varying disturbance with error handling"""
            try:
                delta = params.disturbance_delta(t_now)
                # Ensure proper shape
                if delta.shape[0] != 6:
                    raise ValueError(f"Disturbance must be 6-dimensional, got {delta.shape}")
                return delta.reshape(-1, 1)
            except Exception as e: # pylint: disable=broad-exception-caught
                logger.warning("Disturbance function failed at t=%.3f: %s", t_now, e)
                return np.zeros((6, 1))

        # Set parameter template with all required parameters
        p_template = simulator.get_p_template()
        p_template['pos_ref'] = np.zeros((3, 1))
        p_template['att_ref'] = np.zeros((3, 1))
        p_template['vel_ref'] = np.zeros((3, 1))
        p_template['omega_ref'] = np.zeros((3, 1))
        p_template['disturbance'] = disturbance_func

        def p_fun(t_now):
            """Enhanced parameter function with validation"""
            try:
                # Update disturbance for current time
                p_template['disturbance'] = disturbance_func(t_now)
                return p_template
            except Exception as e: # pylint: disable=broad-exception-caught
                logger.error("Parameter function failed at t=%.3f: %s", t_now, e)
                # Return safe default parameters
                safe_template = simulator.get_p_template()
                for key in safe_template.keys():
                    if 'ref' in key:
                        safe_template[key] = np.zeros((3, 1))
                    elif key == 'disturbance':
                        safe_template[key] = np.zeros((6, 1))
                return safe_template

        simulator.set_p_fun(p_fun)

        # Complete Simulator setup with validation
        try:
            simulator.setup()
            logger.info("do-mpc Simulator setup completed successfully")
        except Exception as e: # pylint: disable=broad-exception-caught
            logger.error("Failed to setup do-mpc Simulator: %s", e)
            raise RuntimeError(f"Simulator setup failed: {e}") # pylint: disable=raise-missing-from

        return simulator

    def _setup_initial_conditions(self):
        """Set initial conditions"""
        # Set initial state
        x0 = params.X0.copy()

        # Stricter numerical cleaning
        # x0 = np.nan_to_num(x0, nan=0.0, posinf=10.0, neginf=-10.0)

        # Limit initial angles to avoid singularities
        # x0[3:6] = np.clip(x0[3:6], -np.pi/2, np.pi/2)

        # Reconstruct state vector to do-mpc expected format
        _x0 = np.concatenate([
            x0[0:3].reshape(-1, 1),   # pos
            x0[3:6].reshape(-1, 1),   # att
            x0[6:9].reshape(-1, 1),   # vel
            x0[9:12].reshape(-1, 1)   # omega
        ])

        # Set initial states for all components
        self.mpc.x0 = _x0
        self.estimator.x0 = _x0
        if self.simulator is not None:
            self.simulator.x0 = _x0

        # Improved initial guess
        try:
            # Set conservative initial control guess
            u0 = np.array([[5.0], [0.0], [0.0]])  # Stable thrust, zero moment

            # Set initial guess for entire prediction horizon
            for k in range(self.mpc.settings.n_horizon):
                self.mpc.u0[k] = u0
                self.mpc.x0[k+1] = _x0  # Keep states stable

            self.mpc.set_initial_guess()

        except Exception as e: # pylint: disable=broad-exception-caught
            print(f"Failed to set initial guess: {e}")

    def step(self, current_state, reference_trajectory, t_current=0.0):
        """
        Execute one step of MPC control

        Args:
            current_state: Current state [12x1]
            reference_trajectory: Reference trajectory dictionary [position, attitude, velocity, angular_velocity]
            t_current: Current time

        Returns:
            control_input: Control input [T, mu, nu]
        """
        _ = t_current
        try:
            # Update current state
            current_x = np.concatenate([
                current_state[0:3].reshape(-1, 1),   # pos
                current_state[3:6].reshape(-1, 1),   # att
                current_state[6:9].reshape(-1, 1),   # vel
                current_state[9:12].reshape(-1, 1)   # omega
            ])

            # Check input validity
            if np.any(np.isnan(current_x)) or np.any(np.isinf(current_x)):
                print("Warning: Input state contains NaN or infinite values")
                return np.array([5.0, 0.0, 0.0])

            # Update reference trajectory parameters
            reference_params = {
                'pos_ref': reference_trajectory['position'].reshape(-1, 1),
                'att_ref': reference_trajectory['attitude'].reshape(-1, 1),
                'vel_ref': reference_trajectory['velocity'].reshape(-1, 1),
                'omega_ref': reference_trajectory['angular_velocity'].reshape(-1, 1)
            }

            # Add disturbance parameters
            reference_params['disturbance'] = np.zeros((6, 1))

            # Set reference trajectory
            self.mpc.set_uncertainty_values(**reference_params)

            # Disturbance compensation
            if self.use_disturbance_compensation:
                self._update_disturbance_compensation(current_state, reference_trajectory)

            # Execute MPC solution
            u_mpc = self.mpc.make_step(current_x)

            # Extract control input
            control_input = self._extract_control_input(u_mpc)

            # Save control input
            self.last_control = control_input

            return control_input

        except Exception as e: # pylint: disable=broad-exception-caught
            print(f"MPC step failed: {e}")
            # Return safe default control
            safe_control = np.array([5.0, 0.0, 0.0])
            self.last_control = safe_control
            return safe_control

    def _update_disturbance_compensation(self, current_state, reference_trajectory):
        """Update disturbance compensation"""
        # Calculate errors
        pos_error = current_state[0:3] - reference_trajectory['position']
        att_error = current_state[3:6] - reference_trajectory['attitude']
        vel_error = current_state[6:9] - reference_trajectory['velocity']
        ang_error = current_state[9:12] - reference_trajectory['angular_velocity']

        e1 = np.concatenate([pos_error, att_error])
        e2 = np.concatenate([vel_error, ang_error])

        # Update disturbance estimation
        gamma = current_state[3:6]
        tau = thrust_params_to_force_torque(self.last_control, self.params.rp_r, self.params.rp_l)

        # Update disturbance observer
        delta_hat = self.disturbance_observer.update(params.DT, e1, e2, tau, gamma)
        self.last_disturbance_estimate = delta_hat

    def _extract_control_input(self, u_mpc):
        """Extract control input"""
        try:
            if hasattr(u_mpc, 'full'):
                u_array = u_mpc.full().flatten()
                control_input = np.array([
                    float(u_array[0]),  # T
                    float(u_array[1]),  # mu
                    float(u_array[2])   # nu
                ])
            elif isinstance(u_mpc, np.ndarray):
                control_input = np.array([
                    float(u_mpc[0]),
                    float(u_mpc[1]),
                    float(u_mpc[2])
                ])
            else:
                u_flat = np.array(u_mpc).flatten()
                control_input = np.array([
                    float(u_flat[0]),
                    float(u_flat[1]),
                    float(u_flat[2])
                ])
        except (IndexError, ValueError, TypeError):
            print("Warning: Unable to properly extract control input")
            control_input = np.array([5.0, 0.0, 0.0])

        # Check validity and limit range
        if np.any(np.isnan(control_input)) or np.any(np.isinf(control_input)):
            print("Warning: Control input contains NaN or infinite values")
            return np.array([5.0, 0.0, 0.0])

        control_input[0] = np.clip(control_input[0], params.T_MIN, params.T_MAX)
        control_input[1] = np.clip(control_input[1], params.MU_MIN, params.MU_MAX)
        control_input[2] = np.clip(control_input[2], params.NU_MIN, params.NU_MAX)

        return control_input

    def get_prediction(self):
        """Get MPC prediction results"""
        try:
            if hasattr(self.mpc, 'data') and self.mpc.data is not None:
                # Use public interface to get prediction data
                prediction_data = {
                    'states': self.mpc.data.prediction(('_x', 'pos')),
                    'controls': self.mpc.data.prediction(('_u', 'T'))
                }
                return prediction_data
            else:
                return {'states': None, 'controls': None}
        except Exception: # pylint: disable=broad-exception-caught
            return {'states': None, 'controls': None}



    def get_current_disturbance_estimate(self):
        """Get current disturbance estimate"""
        if self.use_disturbance_compensation and self.last_disturbance_estimate is not None:
            try:
                # Handle CasADi DM objects
                if hasattr(self.last_disturbance_estimate, 'full'):
                    return self.last_disturbance_estimate.full().flatten()
                # Handle numpy arrays
                elif hasattr(self.last_disturbance_estimate, 'flatten'):
                    return self.last_disturbance_estimate.flatten()
                # Handle other types
                else:
                    return np.array(self.last_disturbance_estimate).flatten()
            except (AttributeError, ValueError):
                return np.zeros(6)
        else:
            return np.zeros(6)

    def reset(self):
        """Reset controller"""
        if self.use_disturbance_compensation and self.disturbance_observer is not None:
            self.disturbance_observer.reset()
            self.last_disturbance_estimate = np.zeros(6)

        # Reset initial conditions
        self._setup_initial_conditions()

        # Clear history data
        self.mpc.reset_history()
        self.estimator.reset_history()
        if self.simulator is not None:
            self.simulator.reset_history()


2. 对比绘图
位置跟踪对比：实际位置 vs 参考位置
姿态跟踪对比：实际姿态 vs 参考姿态
速度跟踪对比：实际速度 vs 参考速度
角速度跟踪对比：实际角速度 vs 参考角速度
3. 新增图表
2D轨迹对比：XY平面上的轨迹对比
3D立体轨迹图：三维空间中的轨迹对比
跟踪误差图：位置误差和姿态误差随时间变化
4. 视觉效果优化
实际轨迹用实线，参考轨迹用虚线
不同的颜色和透明度区分
添加图例和网格线
这样修改后，您可以清楚地看到：

MPC控制器的跟踪性能
实际轨迹与参考轨迹的偏差
螺旋轨迹的完整形状
各个状态变量的跟踪效果